# 04 — MES Trade Processing

## Purpose

This notebook is the technical report and test runner for turning immutable
Databento MES trades into a durable, live-compatible 1-second foundation layer.
It explains the design decisions, retains meaningful boundary tests, and shows
the deliberately gated production command. It does **not** run a production
build by default.

## How to read this notebook

The early sample and full-history validation sections are a preserved learning
record of how the design was evaluated. They are not the current production
path and should not be used to rebuild data. The authoritative executable
implementation begins in the V2 section and lives in
`src/trade_processing_v2.py`.

## Scope and current decision

The completed deliverable in this stage is the 1-second tape dataset. One-minute
and minute-by-price layers remain future research work, not outputs currently
created by this notebook. V2 production and its independent post-run audit
have passed, so the next stage is feature engineering on the approved V2 data.

## Non-negotiable time policy

Databento DataFrames use `ts_recv` as their index, but `ts_event` is exchange
event time. Market bars, CME session dates, first/last trade timestamps, and
the corrected v2 output all use `ts_event`, stored in UTC. Receive time is not
wrong data—it answers a different latency question—and v1 is retained only for
provenance.

## Architecture retained from v1

The v2 processor still reads bounded chunks, carries tick-rule and price-change
state, holds the final incomplete second, resets at contract changes, buffers a
single CME session, validates it, and writes one Parquet file. The changes are
correctness and provenance improvements, not a redesign.

## Open the Raw MES Trade Dataset

The raw Databento DBN file is the immutable source dataset for this project.

In this section, the file is opened and its schema is verified before any aggregation or transformation is performed. Opening the DBNStore does not load all approximately 99.3 million trades into memory at once.

The expected schema is `trades`.

In [1]:
# Import Databento and open the full raw MES trades file.
# This does not load all 99 million records into memory.
import databento as db

raw_file_path = "../data/mes_trades_2025-10-07_to_2026-09-11.dbn"

raw_data = db.DBNStore.from_file(raw_file_path)

print(raw_data)

<DBNStore(schema=trades)>


## CME Trading Session Definition

MES trades nearly 24 hours per day, so a calendar date cannot be treated as the same thing as a trading session.

Before aggregating the raw trades, the processing pipeline must define how each trade will be assigned to a CME trading session. This session structure will later support calculations such as session VWAP, session high and low, time-of-session features, and chronological train/validation/test boundaries.

Session handling must account for weekends, holidays, shortened trading sessions, and daylight saving time rather than assuming every calendar day contains a normal session.

### Session Convention

For this project, MES trading sessions will be defined in the exchange's local timezone, `America/Chicago` (Central Time).

Under the normal CME Globex schedule, an MES trading session begins at **5:00 p.m. CT** and ends at **4:00 p.m. CT on the following calendar day**, followed by the daily maintenance period.

Each session will be labeled by the calendar date on which the session ends. For example, trades occurring from Sunday evening through Monday afternoon belong to the **Monday trading session**.

Using the `America/Chicago` timezone allows daylight saving time to be handled automatically rather than relying on a fixed UTC offset.

Holiday and shortened sessions may differ from the normal schedule. The V2 `is_complete_session` manifest field only means the raw source spans the nominal 5:00 p.m.–4:00 p.m. CT window; it does not claim ordinary exchange hours.

### Test Session Assignment Logic

Before applying session labels to the full dataset, the session-assignment rule will be tested using a small set of timestamps around the normal 5:00 p.m. CT session boundary.

A session is labeled by the date on which it ends. Therefore, a trade occurring at or after 5:00 p.m. CT belongs to the following trading date, while a trade before the daily close belongs to the current trading date.

This small test verifies the basic date-assignment logic and daylight-saving-aware timezone handling before it is incorporated into the full processing pipeline.

In [2]:
# Test the basic CME session-labeling rule on a few example timestamps.
# America/Chicago automatically handles CST/CDT daylight saving changes.
import pandas as pd

test_times = pd.to_datetime([
    "2026-06-15 15:59:00",
    "2026-06-15 16:00:00",
    "2026-06-15 16:30:00",
    "2026-06-15 16:59:00",
    "2026-06-15 17:00:00",
    "2026-06-15 17:01:00",
]).tz_localize("America/Chicago")

session_test = pd.DataFrame({
    "timestamp_ct": test_times
})

# Trades at or after 5:00 p.m. CT belong to the following trading date.
session_test["session_date"] = (
    session_test["timestamp_ct"]
    + pd.Timedelta(hours=7)
).dt.date

session_test

## Historical Architecture Exploration and Future Layers

This historical exploration considered three complementary processed datasets. The completed V2 implementation persists only the approved 1-second foundation; the one-minute and minute-by-price layers remain future feature-engineering work.

### 1. One-Second Tape Dataset

The 1-second dataset preserves short-term trade and tape behavior.

Each persisted V2 active second contains aggregated information such as:

- timestamp
- CME session date
- instrument/contract identifier
- open, high, low, and close price
- total traded volume
- trade count
- maximum trade size
- sum of squared trade sizes
- price × volume sum
- number of distinct traded prices
- first and last trade timestamps within the second
- tick-direction statistics derived from price changes
- Databento native aggressor-side statistics for research
- contract-roll flag, with degraded-data information retained as session-manifest metadata

This layer preserves enough within-minute detail to study trade intensity, bursts of activity, tape speed, short-term price behavior, and other microstructure features later.

### 2. One-Minute Market Dataset

The 1-minute dataset will serve as the primary foundation for feature engineering and modeling.

Each active minute will preserve market statistics including:

- open, high, low, and close price
- total traded volume
- trade count
- VWAP
- trade-size statistics
- number of active seconds
- number of distinct traded prices
- short-term trade-activity summaries
- tick-direction statistics
- Databento native aggressor-side statistics for research
- session, contract, data-quality, and roll information

Rolling indicators, technical indicators, prediction targets, and model-specific features will be created later rather than stored directly in this processing layer.

### 3. Minute-by-Price Dataset

The minute-by-price dataset will preserve where trading occurred inside each 1-minute interval.

Each row will represent a price level that traded during a specific minute and will contain information such as:

- minute timestamp
- CME session date
- instrument/contract identifier
- traded price
- total volume at that price
- number of trades at that price
- Databento native buyer-aggressor volume
- Databento native seller-aggressor volume
- unspecified-side volume

This layer preserves volume-at-price and footprint-style information that would otherwise be lost when individual trades are aggregated into conventional time bars.

## Design Principle

These processed datasets are intended to preserve objective market information, not to define the final model features.

Feature selection, rolling lookback periods, technical indicators, inferred order-flow measures, prediction horizons, model fitting, and trading thresholds will be handled in later stages of the project.

The raw DBN file remains the immutable source of truth.

## Load a Small Trade Sample

Before building the full processing pipeline, a small chunk of raw trades will be loaded into memory and inspected.

This allows the processing logic to be developed and validated on a manageable sample before it is applied incrementally to the full approximately 99.3 million-record dataset.

The sample is used only for pipeline development and does not modify the raw DBN file.

In [3]:
# Load only the first 100,000 raw trade records for processing tests.
# DBNStore.to_df(count=...) returns an iterator, allowing the file to be read in manageable chunks.
trade_chunks = raw_data.to_df(
    count=100_000,
    tz="America/Chicago"
)

sample_trades = next(iter(trade_chunks))

print(f"Sample records: {len(sample_trades):,}")
print(f"First timestamp: {sample_trades.index.min()}")
print(f"Last timestamp:  {sample_trades.index.max()}")

sample_trades.head()

Sample records: 100,000
First timestamp: 2025-10-06 19:00:00.044224940-05:00
Last timestamp:  2025-10-07 09:48:36.836987400-05:00


## Historical Prototype: Prepare Trade Timestamps

The raw trade records contain both `ts_recv` and `ts_event`.

For market aggregation, `ts_event` will be used as the primary timestamp because it represents when the trade event occurred at the exchange. The DataFrame's `ts_recv` index represents when the event was received by Databento and will remain available in the immutable raw dataset but will not define the historical market bars.

For the historical prototype below, this step creates:

- `timestamp_second` — the exchange event timestamp floored to its 1-second interval.
- `timestamp_minute` — a prototype-only exchange event timestamp floored to 1 minute.
- `session_date` — the CME trading-session label using the previously tested 5:00 p.m. Central Time session convention.

The authoritative V2 module persists only `timestamp_second` and session metadata; later layers can derive minute features from the approved 1-second foundation.

In [4]:
# Create a working copy so the original sample remains unchanged.
trades_work = sample_trades.copy()

# ts_event is the exchange event timestamp and will define our market-time aggregations.
trades_work["timestamp_second"] = trades_work["ts_event"].dt.floor("s")
trades_work["timestamp_minute"] = trades_work["ts_event"].dt.floor("min")

# Assign each trade to the CME trading date.
# Adding seven hours moves the 5:00 p.m. CT session boundary to midnight,
# allowing the resulting calendar date to represent the session's ending date.
trades_work["session_date"] = (
    trades_work["ts_event"] + pd.Timedelta(hours=7)
).dt.date

# Inspect the fields that will drive the three processed datasets.
trades_work[
    [
        "ts_event",
        "timestamp_second",
        "timestamp_minute",
        "session_date",
        "instrument_id",
        "symbol",
        "price",
        "size",
        "side",
    ]
].head()

## Build a Sample One-Second Tape Dataset

The first processed layer aggregates individual executions into one record for each active second.

Trades are grouped by their 1-second timestamp and contract identity. The resulting records preserve objective price, volume, trade-size, and native aggressor-side information from the underlying executions.

This sample aggregation is performed only on the 100,000-trade development sample. Its purpose is to verify the calculations and output structure before the same logic is adapted for incremental processing of the full dataset.

Several underlying sums are preserved in addition to derived statistics. This makes it possible to correctly combine or transform the processed data later without returning to the individual trade records.

In [5]:
# Create helper columns needed for the 1-second aggregation.
# These preserve underlying sums that can be reused for later calculations.

# Price multiplied by contracts traded is the numerator used to calculate VWAP.
trades_work["price_x_size"] = trades_work["price"] * trades_work["size"]

# Squared trade size lets us calculate trade-size variance later.
trades_work["size_squared"] = trades_work["size"] ** 2

# Preserve Databento's native aggressor-side volume for research and validation.
# A = seller aggressor, B = buyer aggressor, N = side not specified.
trades_work["native_buy_volume"] = trades_work["size"].where(
    trades_work["side"] == "B", 0
)

trades_work["native_sell_volume"] = trades_work["size"].where(
    trades_work["side"] == "A", 0
)

trades_work["native_unspecified_volume"] = trades_work["size"].where(
    trades_work["side"] == "N", 0
)

# Group all trades occurring in the same second and contract.
second_sample = (
    trades_work
    .groupby(
        [
            "timestamp_second",
            "session_date",
            "instrument_id",
            "symbol",
        ],
        sort=True,
    )
    .agg(
        open_price=("price", "first"),
        high_price=("price", "max"),
        low_price=("price", "min"),
        close_price=("price", "last"),
        volume=("size", "sum"),
        trade_count=("size", "size"),
        trade_size_sum_squared=("size_squared", "sum"),
        max_trade_size=("size", "max"),
        price_x_volume_sum=("price_x_size", "sum"),
        distinct_prices=("price", "nunique"),
        first_trade_time=("ts_event", "first"),
        last_trade_time=("ts_event", "last"),
        native_buy_volume=("native_buy_volume", "sum"),
        native_sell_volume=("native_sell_volume", "sum"),
        native_unspecified_volume=("native_unspecified_volume", "sum"),
    )
    .reset_index()
)

# Calculate statistics that depend on the aggregated totals.
second_sample["average_trade_size"] = (
    second_sample["volume"] / second_sample["trade_count"]
)

second_sample["vwap"] = (
    second_sample["price_x_volume_sum"] / second_sample["volume"]
)

print(f"Raw trades in sample: {len(trades_work):,}")
print(f"Active seconds created: {len(second_sample):,}")

second_sample.head()

Raw trades in sample: 100,000
Active seconds created: 20,682


## Infer Live-Compatible Trade Direction

Databento provides a native aggressor-side field for historical trades, but the planned IBKR live trade feed does not provide an equivalent exchange-native aggressor-side classification.

To maintain historical-to-live compatibility, a second trade-direction measure will therefore be derived using only the sequence of traded prices. This inferred direction can later be calculated using the same logic on both Databento historical trades and IBKR live trades.

The initial classifier uses a tick-rule approach:

- a trade above the previous traded price is classified as buyer-directed
- a trade below the previous traded price is classified as seller-directed
- a trade at the same price inherits the most recent non-zero direction
- direction is unknown until a price change establishes an initial direction

Multiplying inferred direction by trade size produces signed volume. Aggregating signed volume through time creates inferred delta and, later, cumulative delta.

Databento's native aggressor side will be retained separately as a research benchmark. The inferred classifier will be evaluated against the native labels before inferred order-flow features are trusted in the production modeling pipeline.

When the full DBN file is processed incrementally, the last traded price and last established direction from each chunk must be carried into the next chunk so artificial chunk boundaries do not reset the classifier.

In [6]:
# Create a live-compatible trade-direction estimate from the sequence of trade prices.
# +1 = inferred buyer direction
# -1 = inferred seller direction
#  0 = direction not yet known

# Compare each trade price with the immediately preceding trade price.
price_change = trades_work["price"].diff()

# Assign direction only when price actually changes.
trades_work["inferred_direction"] = 0
trades_work.loc[price_change > 0, "inferred_direction"] = 1
trades_work.loc[price_change < 0, "inferred_direction"] = -1

# For trades at the same price, carry forward the most recent non-zero direction.
# Replace zeros with missing values temporarily so forward-fill can propagate direction.
trades_work["inferred_direction"] = (
    trades_work["inferred_direction"]
    .replace(0, pd.NA)
    .ffill()
    .fillna(0)
    .astype("int8")
)

# Convert inferred direction into signed contract volume.
# Positive volume represents inferred buying pressure;
# negative volume represents inferred selling pressure.
trades_work["inferred_signed_volume"] = (
    trades_work["inferred_direction"] * trades_work["size"]
)

# Display a small sequence so we can inspect how price changes create direction labels.
trades_work[
    [
        "ts_event",
        "price",
        "size",
        "side",
        "inferred_direction",
        "inferred_signed_volume",
    ]
].head(20)

## Validate Inferred Trade Direction

Databento provides the exchange-derived aggressor side for historical trades (`B` = buyer aggressor and `A` = seller aggressor). Our live IBKR feed will not provide this same field, so we need to determine whether trade direction can be inferred reliably from price movement alone.

We first compare our tick-rule classification against Databento's known aggressor side on the 100,000-trade sample. If the method performs well, we will repeat the validation across the full historical dataset using memory-safe chunked processing.

Trades for which our classifier has not yet established a direction (`inferred_direction = 0`) are excluded from the accuracy calculation rather than counted as correct or incorrect.

In [9]:
# Convert Databento's known aggressor side into the same numeric format
# as our inferred direction: buyer aggressor = +1, seller aggressor = -1.
trades_work["actual_direction"] = trades_work["side"].map({
    "B": 1,
    "A": -1
})

# Only evaluate trades where:
# 1. Databento provides a known aggressor side (A or B), and
# 2. our tick-rule classifier has established a direction.
validation_sample = trades_work[
    trades_work["actual_direction"].notna()
    & (trades_work["inferred_direction"] != 0)
].copy()

# Compare our inferred direction with Databento's known direction.
validation_sample["correct"] = (
    validation_sample["inferred_direction"]
    == validation_sample["actual_direction"]
)

# Calculate the overall accuracy of the classifier.
accuracy = validation_sample["correct"].mean()

print(f"Trades evaluated: {len(validation_sample):,}")
print(f"Correct classifications: {validation_sample['correct'].sum():,}")
print(f"Incorrect classifications: {(~validation_sample['correct']).sum():,}")
print(f"Tick-rule accuracy: {accuracy:.2%}")

Trades evaluated: 99,995
Correct classifications: 77,571
Incorrect classifications: 22,424
Tick-rule accuracy: 77.57%


## Validate Inferred Direction Across the Full Historical Dataset

The tick-rule classifier will now be evaluated across the entire historical MES trade dataset rather than only the 100,000-trade development sample.

The raw DBN file will be processed incrementally in 500,000-trade chunks so the full approximately 99.3 million records never need to be held in memory at once.

The validation will measure:

- trade-count accuracy
- volume-weighted accuracy
- total correct and incorrect classifications
- total correctly and incorrectly classified contract volume

The final trade price and inferred direction from each chunk will be carried into the next chunk so chunk boundaries do not artificially reset the classifier.

In [10]:
# Validate the tick-rule classifier across the full historical MES dataset.
# Only one 500,000-trade chunk is held in memory at a time.

# Reopen the raw DBN file so validation starts cleanly from the beginning.
validation_data = db.DBNStore.from_file(raw_file_path)

# Read the dataset in 500,000-trade chunks.
chunk_iterator = validation_data.to_df(
    count=500_000,
    tz="America/Chicago"
)

# Running totals across every chunk.
total_raw_trades = 0
total_evaluated_trades = 0
total_correct_trades = 0
total_incorrect_trades = 0
total_evaluated_volume = 0
total_correct_volume = 0
total_incorrect_volume = 0

# Preserve classifier state across chunk boundaries.
previous_price = None
previous_direction = 0

chunk_number = 0

for chunk in chunk_iterator:
    chunk_number += 1
    total_raw_trades += len(chunk)

    # Compare each trade with the immediately preceding trade price.
    price_change = chunk["price"].diff()

    # The first trade of a new chunk must be compared with the final
    # trade price from the previous chunk.
    if previous_price is not None:
        price_change.iloc[0] = chunk["price"].iloc[0] - previous_price

    # Start with unknown inferred direction.
    inferred_direction = pd.Series(
        pd.NA,
        index=chunk.index,
        dtype="Int8"
    )

    # Price increase = inferred buyer direction.
    inferred_direction.loc[price_change > 0] = 1

    # Price decrease = inferred seller direction.
    inferred_direction.loc[price_change < 0] = -1

    # If the first trade has unchanged price, inherit the previous
    # chunk's final established direction.
    if previous_direction != 0 and pd.isna(inferred_direction.iloc[0]):
        inferred_direction.iloc[0] = previous_direction

    # Same-price trades inherit the most recent known direction.
    inferred_direction = (
        inferred_direction
        .ffill()
        .fillna(0)
        .astype("int8")
    )

    # Convert Databento native aggressor side to numeric direction.
    actual_direction = chunk["side"].map({
        "B": 1,
        "A": -1
    })

    # Evaluate only known A/B trades where our classifier has a direction.
    valid = (
        actual_direction.notna()
        & (inferred_direction != 0)
    )

    inferred_valid = inferred_direction[valid]
    actual_valid = actual_direction[valid]
    size_valid = chunk.loc[valid, "size"]

    correct = inferred_valid == actual_valid

    # Update trade-count totals.
    evaluated_count = int(valid.sum())
    correct_count = int(correct.sum())

    total_evaluated_trades += evaluated_count
    total_correct_trades += correct_count
    total_incorrect_trades += evaluated_count - correct_count

    # Update contract-volume totals.
    evaluated_volume = int(size_valid.sum())
    correct_volume = int(size_valid[correct].sum())

    total_evaluated_volume += evaluated_volume
    total_correct_volume += correct_volume
    total_incorrect_volume += evaluated_volume - correct_volume

    # Carry market state into the next chunk.
    previous_price = chunk["price"].iloc[-1]
    previous_direction = int(inferred_direction.iloc[-1])

    # Print progress every 20 chunks.
    if chunk_number % 20 == 0:
        print(
            f"Chunks processed: {chunk_number:,} | "
            f"Raw trades processed: {total_raw_trades:,}"
        )

# Calculate final full-history agreement rates.
trade_accuracy = total_correct_trades / total_evaluated_trades
volume_accuracy = total_correct_volume / total_evaluated_volume

print("\nFULL-DATASET TICK-RULE VALIDATION")
print("---------------------------------")
print(f"Raw trades processed:        {total_raw_trades:,}")
print(f"Trades evaluated:            {total_evaluated_trades:,}")
print(f"Correct classifications:     {total_correct_trades:,}")
print(f"Incorrect classifications:   {total_incorrect_trades:,}")
print(f"Trade-count accuracy:        {trade_accuracy:.2%}")
print()
print(f"Contract volume evaluated:   {total_evaluated_volume:,}")
print(f"Correctly classified volume: {total_correct_volume:,}")
print(f"Incorrectly classified vol.: {total_incorrect_volume:,}")
print(f"Volume-weighted accuracy:    {volume_accuracy:.2%}")

Chunks processed: 20 | Raw trades processed: 10,000,000
Chunks processed: 40 | Raw trades processed: 20,000,000
Chunks processed: 60 | Raw trades processed: 30,000,000
Chunks processed: 80 | Raw trades processed: 40,000,000
Chunks processed: 100 | Raw trades processed: 50,000,000
Chunks processed: 120 | Raw trades processed: 60,000,000
Chunks processed: 140 | Raw trades processed: 70,000,000
Chunks processed: 160 | Raw trades processed: 80,000,000
Chunks processed: 180 | Raw trades processed: 90,000,000

FULL-DATASET TICK-RULE VALIDATION
---------------------------------
Raw trades processed:        99,319,450
Trades evaluated:            99,319,172
Correct classifications:     74,148,006
Incorrect classifications:   25,171,166
Trade-count accuracy:        74.66%

Contract volume evaluated:   298,520,236
Correctly classified volume: 229,585,607
Incorrectly classified vol.: 68,934,629
Volume-weighted accuracy:    76.91%


## Analyze Tick-Rule Accuracy by Price Change

The full-history validation showed that the simple tick-rule classifier agrees with Databento's native aggressor side on approximately 74.7% of trades and 76.9% of contract volume.

The next diagnostic examines whether classification accuracy differs between:

- **price-changing trades** — the current trade occurs above or below the immediately preceding trade price
- **same-price trades** — the current trade occurs at the same price and therefore inherits the most recent non-zero tick direction

This distinction is important because same-price trades rely on carried-forward direction rather than a new observable price movement and may account for a large portion of classification errors.

The analysis will again use the full historical dataset and preserve classifier state across processing chunks.

In [11]:
# Measure tick-rule accuracy separately for price-changing and same-price trades
# across the full historical MES dataset.

diagnostic_data = db.DBNStore.from_file(raw_file_path)

chunk_iterator = diagnostic_data.to_df(
    count=500_000,
    tz="America/Chicago"
)

# Store running totals for the two trade types.
results = {
    "price_changed": {
        "trades": 0,
        "correct": 0,
        "volume": 0,
        "correct_volume": 0,
    },
    "same_price": {
        "trades": 0,
        "correct": 0,
        "volume": 0,
        "correct_volume": 0,
    },
}

previous_price = None
previous_direction = 0
total_raw_trades = 0
chunk_number = 0

for chunk in chunk_iterator:
    chunk_number += 1
    total_raw_trades += len(chunk)

    # Compare each trade price with the immediately preceding trade.
    price_change = chunk["price"].diff()

    # Continue the price sequence across chunk boundaries.
    if previous_price is not None:
        price_change.iloc[0] = chunk["price"].iloc[0] - previous_price

    # Create the tick-rule direction.
    inferred_direction = pd.Series(
        pd.NA,
        index=chunk.index,
        dtype="Int8"
    )

    inferred_direction.loc[price_change > 0] = 1
    inferred_direction.loc[price_change < 0] = -1

    # Continue the previous direction across a chunk boundary when needed.
    if previous_direction != 0 and pd.isna(inferred_direction.iloc[0]):
        inferred_direction.iloc[0] = previous_direction

    inferred_direction = (
        inferred_direction
        .ffill()
        .fillna(0)
        .astype("int8")
    )

    # Convert Databento native side to numeric direction.
    actual_direction = chunk["side"].map({
        "B": 1,
        "A": -1
    })

    # Only use trades with a known native side and an established inferred direction.
    valid = (
        actual_direction.notna()
        & (inferred_direction != 0)
    )

    # Separate trades according to whether price actually moved.
    changed_mask = valid & (price_change != 0)
    same_mask = valid & (price_change == 0)

    for label, mask in [
        ("price_changed", changed_mask),
        ("same_price", same_mask),
    ]:
        inferred_subset = inferred_direction[mask]
        actual_subset = actual_direction[mask]
        size_subset = chunk.loc[mask, "size"]

        correct = inferred_subset == actual_subset

        results[label]["trades"] += int(mask.sum())
        results[label]["correct"] += int(correct.sum())
        results[label]["volume"] += int(size_subset.sum())
        results[label]["correct_volume"] += int(
            size_subset[correct].sum()
        )

    # Preserve state for the next chunk.
    previous_price = chunk["price"].iloc[-1]
    previous_direction = int(inferred_direction.iloc[-1])

    # Show progress every 20 chunks.
    if chunk_number % 20 == 0:
        print(
            f"Chunks processed: {chunk_number:,} | "
            f"Raw trades processed: {total_raw_trades:,}"
        )

# Display the final accuracy for each trade type.
print("\nTICK-RULE ACCURACY BY PRICE CHANGE")
print("----------------------------------")

for label in ["price_changed", "same_price"]:
    trade_accuracy = (
        results[label]["correct"] / results[label]["trades"]
    )

    volume_accuracy = (
        results[label]["correct_volume"] / results[label]["volume"]
    )

    print(f"\n{label.upper()}")
    print(f"Trades evaluated:         {results[label]['trades']:,}")
    print(f"Correct classifications:  {results[label]['correct']:,}")
    print(f"Trade-count accuracy:     {trade_accuracy:.2%}")
    print(f"Contract volume:          {results[label]['volume']:,}")
    print(f"Volume-weighted accuracy: {volume_accuracy:.2%}")

Chunks processed: 20 | Raw trades processed: 10,000,000
Chunks processed: 40 | Raw trades processed: 20,000,000
Chunks processed: 60 | Raw trades processed: 30,000,000
Chunks processed: 80 | Raw trades processed: 40,000,000
Chunks processed: 100 | Raw trades processed: 50,000,000
Chunks processed: 120 | Raw trades processed: 60,000,000
Chunks processed: 140 | Raw trades processed: 70,000,000
Chunks processed: 160 | Raw trades processed: 80,000,000
Chunks processed: 180 | Raw trades processed: 90,000,000

TICK-RULE ACCURACY BY PRICE CHANGE
----------------------------------

PRICE_CHANGED
Trades evaluated:         32,435,094
Correct classifications:  30,081,291
Trade-count accuracy:     92.74%
Contract volume:          109,303,940
Volume-weighted accuracy: 94.80%

SAME_PRICE
Trades evaluated:         66,884,078
Correct classifications:  44,066,715
Trade-count accuracy:     65.89%
Contract volume:          189,216,296
Volume-weighted accuracy: 66.57%


## Analyze Same-Price Run Length

The simple tick rule carries the most recent non-zero price direction forward across trades that occur at the same price.

This section measures whether that carried-forward direction becomes less reliable as more consecutive trades occur without a price change.

For each same-price trade, a `same_price_run_position` value will represent how deep the trade is into the current run of unchanged prices.

Examples:

- first same-price trade after a price move → position 1
- second consecutive same-price trade → position 2
- tenth consecutive same-price trade → position 10

Accuracy will then be measured by run position across the full historical dataset.

If accuracy declines materially as run length increases, the current carry-forward rule may be useful only for the early part of a same-price run and a different classification method may be needed for later trades.

In [14]:
# Measure same-price tick-rule accuracy by position within a run of unchanged prices.
# This version uses positional arrays rather than the Databento ts_recv index,
# because multiple trades can share the same receive timestamp.

run_data = db.DBNStore.from_file(raw_file_path)

chunk_iterator = run_data.to_df(
    count=500_000,
    tz="America/Chicago"
)

# Running results for each same-price run-position bucket.
run_buckets = {
    "1": {"trades": 0, "correct": 0, "volume": 0, "correct_volume": 0},
    "2": {"trades": 0, "correct": 0, "volume": 0, "correct_volume": 0},
    "3": {"trades": 0, "correct": 0, "volume": 0, "correct_volume": 0},
    "4-5": {"trades": 0, "correct": 0, "volume": 0, "correct_volume": 0},
    "6-10": {"trades": 0, "correct": 0, "volume": 0, "correct_volume": 0},
    "11-20": {"trades": 0, "correct": 0, "volume": 0, "correct_volume": 0},
    "21-50": {"trades": 0, "correct": 0, "volume": 0, "correct_volume": 0},
    "51+": {"trades": 0, "correct": 0, "volume": 0, "correct_volume": 0},
}

# Preserve market state across chunk boundaries.
previous_price = None
previous_direction = 0
previous_run_position = 0

total_raw_trades = 0
chunk_number = 0

for chunk in chunk_iterator:
    chunk_number += 1
    total_raw_trades += len(chunk)

    # Compare each trade price with the immediately preceding trade.
    price_change = chunk["price"].diff()

    # Continue the price sequence across chunk boundaries.
    if previous_price is not None:
        price_change.iloc[0] = chunk["price"].iloc[0] - previous_price

    # Build tick-rule direction.
    inferred_direction = pd.Series(
        pd.NA,
        index=range(len(chunk)),
        dtype="Int8"
    )

    price_change_values = price_change.to_numpy()

    inferred_direction.loc[price_change_values > 0] = 1
    inferred_direction.loc[price_change_values < 0] = -1

    # Carry the prior chunk's direction into this chunk if the first trade
    # occurs at the same price.
    if previous_direction != 0 and pd.isna(inferred_direction.iloc[0]):
        inferred_direction.iloc[0] = previous_direction

    inferred_direction = (
        inferred_direction
        .ffill()
        .fillna(0)
        .astype("int8")
        .to_numpy()
    )

    # Convert Databento native aggressor side to numeric direction.
    actual_direction = (
        chunk["side"]
        .map({"B": 1, "A": -1})
        .to_numpy()
    )

    sizes = chunk["size"].to_numpy()

    # Calculate each trade's position within a consecutive same-price run.
    run_positions = []
    current_run_position = previous_run_position

    for change in price_change_values:
        if pd.isna(change):
            current_run_position = 0
        elif change == 0:
            current_run_position += 1
        else:
            current_run_position = 0

        run_positions.append(current_run_position)

    # Evaluate same-price trades positionally, avoiding duplicate-index issues.
    for i in range(len(chunk)):
        position = run_positions[i]

        if (
            position == 0
            or pd.isna(actual_direction[i])
            or inferred_direction[i] == 0
        ):
            continue

        if position == 1:
            bucket = "1"
        elif position == 2:
            bucket = "2"
        elif position == 3:
            bucket = "3"
        elif position <= 5:
            bucket = "4-5"
        elif position <= 10:
            bucket = "6-10"
        elif position <= 20:
            bucket = "11-20"
        elif position <= 50:
            bucket = "21-50"
        else:
            bucket = "51+"

        size = int(sizes[i])
        is_correct = inferred_direction[i] == actual_direction[i]

        run_buckets[bucket]["trades"] += 1
        run_buckets[bucket]["volume"] += size

        if is_correct:
            run_buckets[bucket]["correct"] += 1
            run_buckets[bucket]["correct_volume"] += size

    # Carry the final market state into the next chunk.
    previous_price = chunk["price"].iloc[-1]
    previous_direction = int(inferred_direction[-1])
    previous_run_position = int(run_positions[-1])

    if chunk_number % 20 == 0:
        print(
            f"Chunks processed: {chunk_number:,} | "
            f"Raw trades processed: {total_raw_trades:,}"
        )

print("\nSAME-PRICE ACCURACY BY RUN POSITION")
print("-----------------------------------")

for bucket, stats in run_buckets.items():
    trade_accuracy = stats["correct"] / stats["trades"]
    volume_accuracy = stats["correct_volume"] / stats["volume"]

    print(f"\nRun position: {bucket}")
    print(f"Trades evaluated:         {stats['trades']:,}")
    print(f"Trade-count accuracy:     {trade_accuracy:.2%}")
    print(f"Contract volume:          {stats['volume']:,}")
    print(f"Volume-weighted accuracy: {volume_accuracy:.2%}")

Chunks processed: 20 | Raw trades processed: 10,000,000
Chunks processed: 40 | Raw trades processed: 20,000,000
Chunks processed: 60 | Raw trades processed: 30,000,000
Chunks processed: 80 | Raw trades processed: 40,000,000
Chunks processed: 100 | Raw trades processed: 50,000,000
Chunks processed: 120 | Raw trades processed: 60,000,000
Chunks processed: 140 | Raw trades processed: 70,000,000
Chunks processed: 160 | Raw trades processed: 80,000,000
Chunks processed: 180 | Raw trades processed: 90,000,000

SAME-PRICE ACCURACY BY RUN POSITION
-----------------------------------

Run position: 1
Trades evaluated:         18,555,845
Trade-count accuracy:     75.53%
Contract volume:          53,459,821
Volume-weighted accuracy: 77.56%

Run position: 2
Trades evaluated:         12,365,035
Trade-count accuracy:     65.84%
Contract volume:          35,109,004
Volume-weighted accuracy: 67.46%

Run position: 3
Trades evaluated:         8,729,704
Trade-count accuracy:     60.78%
Contract volume:  

## Building a More Reliable Trade-Direction Classifier

The basic tick rule performs very well when the trade price changes, but it is much less reliable when many consecutive trades occur at the same price.

Our full-dataset validation showed that accuracy depends strongly on where a trade occurs within a same-price run. This means we should not treat every inferred trade direction as equally reliable.

The next step is to use what we learned from this validation to design a more reliable historical trade-direction method. The goal is to preserve useful order-flow information for features such as:

- Delta
- Cumulative delta
- Buying and selling pressure
- Absorption
- Exhaustion
- Effort versus result

Any method we use for production features must also be reproducible from the live IBKR trade stream. Databento's native aggressor side will therefore be used as a historical benchmark for validation, not as information that the live model would directly receive.

In [15]:
# Create a working copy of our 100,000-trade sample for the delta comparison.
# We use trades_work because it already contains our inferred_direction column.
delta_test = trades_work.copy()

# Create Databento's "true" signed volume using its native aggressor-side label.
# B = buyer aggressor, so contract volume is positive.
# A = seller aggressor, so contract volume is negative.
# N = Databento did not specify a side, so signed volume is left unknown.
delta_test["true_signed_volume"] = (
    delta_test["size"]
    * delta_test["side"].map({
        "B": 1,
        "A": -1,
    })
)

# Create our live-compatible inferred signed volume using the tick rule.
# +1 direction produces positive volume and -1 produces negative volume.
delta_test["inferred_signed_volume"] = (
    delta_test["size"]
    * delta_test["inferred_direction"]
)

# Display a small sample so we can verify that both signed-volume
# calculations behave as expected before aggregating them into delta.
delta_test[
    [
        "ts_event",
        "price",
        "size",
        "side",
        "inferred_direction",
        "true_signed_volume",
        "inferred_signed_volume",
    ]
].head(20)

In [16]:
# Aggregate true and inferred signed volume into 1-second intervals.
# This lets us test whether individual trade-classification errors become
# less important when many trades are combined into short time windows.
second_delta = (
    delta_test
    .groupby("timestamp_second")
    .agg(
        true_delta=("true_signed_volume", "sum"),
        inferred_delta=("inferred_signed_volume", "sum"),
        trade_count=("size", "size"),
        total_volume=("size", "sum"),
    )
    .reset_index()
)

# Aggregate the same data into 1-minute intervals.
# The 1-minute level is especially important because our current plan is
# for the eventual model to evaluate market conditions once per minute.
minute_delta = (
    delta_test
    .groupby("timestamp_minute")
    .agg(
        true_delta=("true_signed_volume", "sum"),
        inferred_delta=("inferred_signed_volume", "sum"),
        trade_count=("size", "size"),
        total_volume=("size", "sum"),
    )
    .reset_index()
)

# Display the first 10 one-second intervals so we can verify that
# the aggregation worked correctly before calculating comparison metrics.
second_delta.head(10)

In [17]:
# Measure how closely our live-compatible inferred delta tracks Databento's
# native "true" delta at both the 1-second and 1-minute levels.

# Correlation measures whether true and inferred delta tend to move together.
# A value near 1.0 means they track each other very closely.
second_correlation = second_delta["true_delta"].corr(
    second_delta["inferred_delta"]
)
minute_correlation = minute_delta["true_delta"].corr(
    minute_delta["inferred_delta"]
)

# For sign agreement, keep only intervals where both delta values are nonzero.
# This asks a simpler but important question:
# Did we at least identify buying pressure vs. selling pressure correctly?
second_nonzero = second_delta[
    (second_delta["true_delta"] != 0)
    & (second_delta["inferred_delta"] != 0)
].copy()

minute_nonzero = minute_delta[
    (minute_delta["true_delta"] != 0)
    & (minute_delta["inferred_delta"] != 0)
].copy()

second_sign_agreement = (
    (second_nonzero["true_delta"] > 0)
    == (second_nonzero["inferred_delta"] > 0)
).mean()

minute_sign_agreement = (
    (minute_nonzero["true_delta"] > 0)
    == (minute_nonzero["inferred_delta"] > 0)
).mean()

# Mean absolute error measures the average difference, in contracts,
# between true delta and inferred delta.
second_mae = (
    second_delta["true_delta"] - second_delta["inferred_delta"]
).abs().mean()

minute_mae = (
    minute_delta["true_delta"] - minute_delta["inferred_delta"]
).abs().mean()

# Print the results together so the effect of aggregation is easy to compare.
print("TRUE VS INFERRED DELTA — 100,000-TRADE SAMPLE")
print("------------------------------------------------")
print(f"1-second intervals: {len(second_delta):,}")
print(f"1-second correlation: {second_correlation:.4f}")
print(f"1-second sign agreement: {second_sign_agreement:.2%}")
print(f"1-second mean absolute error: {second_mae:.2f} contracts")
print()
print(f"1-minute intervals: {len(minute_delta):,}")
print(f"1-minute correlation: {minute_correlation:.4f}")
print(f"1-minute sign agreement: {minute_sign_agreement:.2%}")
print(f"1-minute mean absolute error: {minute_mae:.2f} contracts")

TRUE VS INFERRED DELTA — 100,000-TRADE SAMPLE
------------------------------------------------
1-second intervals: 20,682
1-second correlation: 0.7439
1-second sign agreement: 78.85%
1-second mean absolute error: 7.07 contracts

1-minute intervals: 889
1-minute correlation: 0.8630
1-minute sign agreement: 79.59%
1-minute mean absolute error: 47.61 contracts


## Full-History Validation of Inferred Delta

The 100,000-trade sample showed that the simple tick-rule classifier reconstructs aggregated delta much better than its individual trade accuracy might suggest.

The next test applies the same live-compatible classification method to the complete historical dataset of approximately 99.3 million MES trades.

We will evaluate inferred delta at two resolutions:

- **1 second** — preserves short-term tape and order-flow behavior
- **1 minute** — matches the current planned prediction cadence for the modeling system

At both resolutions, we will compare:

- Databento native delta, used only as the historical benchmark
- Tick-rule inferred delta, which can be reproduced from live IBKR trades
- Total traded volume
- Trade count

The main evaluation metrics will be:

- Correlation between native and inferred delta
- Agreement on whether buying or selling pressure dominated
- Mean absolute error in contracts

The raw data will continue to be processed in 500,000-trade chunks so that the full dataset never needs to be loaded into memory at once. Partial seconds and minutes that cross chunk boundaries will be recombined before they are evaluated.

The tick-rule state will also be preserved across chunk boundaries, but it will reset when the active futures contract changes so that a contract-roll price difference is not mistaken for a normal market price movement.

If inferred delta remains reasonably close to native delta across the full historical dataset, we can keep the simple live-compatible classifier and continue building the main processing pipeline instead of prematurely adding a more complex same-price classification model.

In [19]:
# Validate our live-compatible inferred delta against Databento's native
# aggressor-side delta across the complete ~99.3-million-trade dataset.
#
# This pass evaluates BOTH 1-second and 1-minute intervals while keeping
# memory usage low. We retain only running statistics rather than millions
# of aggregated rows.

import math

# Open the immutable raw Databento DBN file and read it in manageable chunks.
full_delta_data = db.DBNStore.from_file(raw_file_path)

chunk_iterator = full_delta_data.to_df(
    count=500_000,
    tz="America/Chicago"
)

# Create an empty statistics container for one aggregation level.
# These accumulated values are enough to calculate correlation,
# directional agreement, and mean absolute error without retaining
# every second/minute in memory.
def create_delta_stats():
    return {
        "intervals": 0,
        "sum_true": 0.0,
        "sum_inferred": 0.0,
        "sum_true_squared": 0.0,
        "sum_inferred_squared": 0.0,
        "sum_product": 0.0,
        "absolute_error": 0.0,
        "sign_intervals": 0,
        "sign_matches": 0,
        "trade_count": 0,
        "total_volume": 0,
    }

second_stats = create_delta_stats()
minute_stats = create_delta_stats()

# Update running statistics using intervals that are known to be complete.
def update_delta_stats(stats, frame):
    if len(frame) == 0:
        return

    true_values = frame["true_delta"].astype("float64")
    inferred_values = frame["inferred_delta"].astype("float64")

    stats["intervals"] += len(frame)
    stats["sum_true"] += true_values.sum()
    stats["sum_inferred"] += inferred_values.sum()
    stats["sum_true_squared"] += (true_values ** 2).sum()
    stats["sum_inferred_squared"] += (inferred_values ** 2).sum()
    stats["sum_product"] += (true_values * inferred_values).sum()
    stats["absolute_error"] += (
        true_values - inferred_values
    ).abs().sum()

    # Sign agreement is evaluated only when both delta values are nonzero.
    sign_mask = (true_values != 0) & (inferred_values != 0)

    stats["sign_intervals"] += int(sign_mask.sum())
    stats["sign_matches"] += int(
        (
            (true_values[sign_mask] > 0)
            == (inferred_values[sign_mask] > 0)
        ).sum()
    )

    stats["trade_count"] += int(frame["trade_count"].sum())
    stats["total_volume"] += int(frame["total_volume"].sum())

# Chunk boundaries can occur halfway through a second or minute.
# We therefore hold the final interval from each chunk until the next
# chunk arrives and combine the two pieces when necessary.
def process_aggregated_intervals(frame, time_column, pending, stats):
    if len(frame) == 0:
        return pending

    frame = frame.sort_values(time_column).reset_index(drop=True)

    # Recombine the previous chunk's final partial interval when the
    # current chunk begins inside that same second/minute.
    if pending is not None:
        if frame.loc[0, time_column] == pending[time_column]:
            for column in [
                "true_delta",
                "inferred_delta",
                "trade_count",
                "total_volume",
            ]:
                frame.loc[0, column] += pending[column]
        else:
            update_delta_stats(
                stats,
                pd.DataFrame([pending])
            )

    # Every row except the final one is now known to be complete.
    if len(frame) > 1:
        update_delta_stats(
            stats,
            frame.iloc[:-1]
        )

    # Hold the final interval because the next raw-data chunk may
    # contain more trades belonging to this same second/minute.
    return frame.iloc[-1].to_dict()

# Preserve tick-rule state across ordinary chunk boundaries.
previous_price = None
previous_direction = 0
previous_instrument = None

# Preserve incomplete time intervals across chunk boundaries.
pending_second = None
pending_minute = None

total_raw_trades = 0
chunk_number = 0

for chunk in chunk_iterator:
    chunk_number += 1
    total_raw_trades += len(chunk)

    # Reset to a simple RangeIndex because Databento receive timestamps
    # are not guaranteed to be unique.
    work = chunk[
        [
            "ts_event",
            "instrument_id",
            "price",
            "size",
            "side",
        ]
    ].reset_index(drop=True)

    # Compare each trade price with the immediately preceding trade price.
    price_change = work["price"].diff()

    # Identify futures contract changes.
    # The tick rule must not interpret a roll from one contract to another
    # as an ordinary price increase or decrease.
    contract_change = work["instrument_id"].ne(
        work["instrument_id"].shift()
    )

    # Correctly connect the first trade of this chunk to the previous chunk
    # only when both trades belong to the same futures contract.
    if previous_instrument == work["instrument_id"].iloc[0]:
        contract_change.iloc[0] = False
        price_change.iloc[0] = (
            work["price"].iloc[0] - previous_price
        )
    else:
        contract_change.iloc[0] = True
        price_change.iloc[0] = pd.NA

    # Remove artificial price changes at every contract-roll boundary.
    price_change.loc[contract_change] = pd.NA

    # Build the live-compatible tick-rule direction.
    # +1 = inferred buyer aggressor
    # -1 = inferred seller aggressor
    #  0 = direction not yet established
    inferred_direction = pd.Series(
        pd.NA,
        index=work.index,
        dtype="Int8"
    )

    inferred_direction.loc[price_change > 0] = 1
    inferred_direction.loc[price_change < 0] = -1

    # If this chunk continues the same contract and begins at the same
    # price as the previous trade, carry the previous known direction.
    if (
        not contract_change.iloc[0]
        and price_change.iloc[0] == 0
        and previous_direction != 0
    ):
        inferred_direction.iloc[0] = previous_direction

    # Create contract segments so forward-filling never crosses a roll.
    contract_segment = contract_change.cumsum()

    inferred_direction = (
        inferred_direction
        .groupby(contract_segment)
        .ffill()
        .fillna(0)
        .astype("int8")
    )

    # Convert Databento's native aggressor side into the historical
    # benchmark direction.
    # B = buyer aggressor (+1)
    # A = seller aggressor (-1)
    # N / unspecified = 0
    true_direction = (
        work["side"]
        .map({
            "B": 1,
            "A": -1,
        })
        .fillna(0)
        .astype("int8")
    )

    # Convert trade direction into signed contract volume.
    true_signed_volume = (
        work["size"] * true_direction
    )

    inferred_signed_volume = (
        work["size"] * inferred_direction
    )

    # Create the two time resolutions we want to evaluate.
    timestamp_second = work["ts_event"].dt.floor("s")
    timestamp_minute = work["ts_event"].dt.floor("min")

    # Build a temporary trade-level frame containing only the information
    # needed for this validation.
    delta_work = pd.DataFrame({
        "timestamp_second": timestamp_second,
        "timestamp_minute": timestamp_minute,
        "true_signed_volume": true_signed_volume,
        "inferred_signed_volume": inferred_signed_volume,
        "size": work["size"],
    })

    # Aggregate this raw-data chunk into 1-second intervals.
    chunk_seconds = (
        delta_work
        .groupby("timestamp_second")
        .agg(
            true_delta=("true_signed_volume", "sum"),
            inferred_delta=("inferred_signed_volume", "sum"),
            trade_count=("size", "size"),
            total_volume=("size", "sum"),
        )
        .reset_index()
    )

    # Aggregate the same trades into 1-minute intervals.
    chunk_minutes = (
        delta_work
        .groupby("timestamp_minute")
        .agg(
            true_delta=("true_signed_volume", "sum"),
            inferred_delta=("inferred_signed_volume", "sum"),
            trade_count=("size", "size"),
            total_volume=("size", "sum"),
        )
        .reset_index()
    )

    # Evaluate complete intervals while safely carrying the final
    # potentially incomplete interval into the next chunk.
    pending_second = process_aggregated_intervals(
        chunk_seconds,
        "timestamp_second",
        pending_second,
        second_stats,
    )

    pending_minute = process_aggregated_intervals(
        chunk_minutes,
        "timestamp_minute",
        pending_minute,
        minute_stats,
    )

    # Preserve the final market state for the next raw-data chunk.
    previous_price = work["price"].iloc[-1]
    previous_instrument = work["instrument_id"].iloc[-1]

    # If the final trade belongs to a contract segment in which direction
    # has been established, preserve it for the next chunk.
    previous_direction = int(inferred_direction.iloc[-1])

    # Print progress approximately every 10 million raw trades.
    if chunk_number % 20 == 0:
        print(
            f"Chunks processed: {chunk_number:,} | "
            f"Raw trades processed: {total_raw_trades:,}"
        )

# The final second and minute in the entire dataset can now be finalized
# because there is no later chunk that could contain additional trades.
if pending_second is not None:
    update_delta_stats(
        second_stats,
        pd.DataFrame([pending_second])
    )

if pending_minute is not None:
    update_delta_stats(
        minute_stats,
        pd.DataFrame([pending_minute])
    )

# Calculate final metrics from the accumulated sufficient statistics.
def calculate_final_metrics(stats):
    n = stats["intervals"]

    numerator = (
        n * stats["sum_product"]
        - stats["sum_true"] * stats["sum_inferred"]
    )

    true_variance_term = (
        n * stats["sum_true_squared"]
        - stats["sum_true"] ** 2
    )

    inferred_variance_term = (
        n * stats["sum_inferred_squared"]
        - stats["sum_inferred"] ** 2
    )

    correlation = numerator / math.sqrt(
        true_variance_term * inferred_variance_term
    )

    sign_agreement = (
        stats["sign_matches"]
        / stats["sign_intervals"]
    )

    mae = (
        stats["absolute_error"]
        / stats["intervals"]
    )

    return correlation, sign_agreement, mae

second_correlation, second_sign_agreement, second_mae = (
    calculate_final_metrics(second_stats)
)

minute_correlation, minute_sign_agreement, minute_mae = (
    calculate_final_metrics(minute_stats)
)

# Print the full-history results in the same format as the sample test
# so we can directly compare whether performance holds across the dataset.
print()
print("TRUE VS INFERRED DELTA — FULL HISTORICAL DATASET")
print("-------------------------------------------------")
print(f"Raw trades processed: {total_raw_trades:,}")
print()
print(f"1-second intervals: {second_stats['intervals']:,}")
print(f"1-second correlation: {second_correlation:.4f}")
print(f"1-second sign agreement: {second_sign_agreement:.2%}")
print(f"1-second mean absolute error: {second_mae:.2f} contracts")
print()
print(f"1-minute intervals: {minute_stats['intervals']:,}")
print(f"1-minute correlation: {minute_correlation:.4f}")
print(f"1-minute sign agreement: {minute_sign_agreement:.2%}")
print(f"1-minute mean absolute error: {minute_mae:.2f} contracts")

Chunks processed: 20 | Raw trades processed: 10,000,000
Chunks processed: 40 | Raw trades processed: 20,000,000
Chunks processed: 60 | Raw trades processed: 30,000,000
Chunks processed: 80 | Raw trades processed: 40,000,000
Chunks processed: 100 | Raw trades processed: 50,000,000
Chunks processed: 120 | Raw trades processed: 60,000,000
Chunks processed: 140 | Raw trades processed: 70,000,000
Chunks processed: 160 | Raw trades processed: 80,000,000
Chunks processed: 180 | Raw trades processed: 90,000,000

TRUE VS INFERRED DELTA — FULL HISTORICAL DATASET
-------------------------------------------------
Raw trades processed: 99,319,450

1-second intervals: 13,165,975
1-second correlation: 0.8033
1-second sign agreement: 75.78%
1-second mean absolute error: 7.98 contracts

1-minute intervals: 329,337
1-minute correlation: 0.8807
1-minute sign agreement: 78.38%
1-minute mean absolute error: 71.98 contracts


## Inferred Delta Decision

The full-history validation confirms that the simple tick-rule classifier is sufficiently reliable for the first version of the live-compatible MES feature pipeline.

Across 99,319,450 historical trades:

- 1-second delta correlation: **0.8033**
- 1-second directional agreement: **75.78%**
- 1-minute delta correlation: **0.8807**
- 1-minute directional agreement: **78.38%**

Although individual trade classifications are imperfect, aggregation substantially preserves the underlying order-flow signal. The strong full-history 1-minute correlation indicates that the simple classifier is adequate for testing whether inferred order-flow features provide incremental predictive value.

Databento's native aggressor side will remain available only as a historical benchmark. Production features will use the live-compatible inferred direction so that the same feature logic can later be reproduced from IBKR trade data.

**Decision:** freeze the simple tick-rule method as `tick_rule_v1` and move forward. A more complex trade-direction classifier will only be reconsidered if later modeling shows that order-flow information is valuable but the live-compatible approximation is materially limiting performance.

## Completed V2 1-Second Foundation

The completed V2 stage converted the raw Databento MES trade history into a reusable event-time 1-second tape dataset.

This layer will sit between the immutable raw trade data and all later feature engineering and modeling:

Raw trades → 1-second tape layer → 1-minute/model features → predictive models

The purpose of the 1-second layer is to preserve the important price, volume, trade-intensity, and live-compatible order-flow information while dramatically reducing the size and processing cost of the raw dataset.

### Design Principles

- Raw Databento DBN data remains immutable.
- UTC will be used as the canonical timestamp for storage.
- CME session dates will be derived using Chicago time.
- Each second will retain enough information to construct later rolling features without returning to individual trades whenever possible.
- Databento native aggressor-side information will be retained only for historical research and validation.
- Production order-flow features will use the frozen `tick_rule_v1` inferred direction so the same logic can later be reproduced from IBKR live trades.
- Futures contract changes are flagged per row; degraded and source-boundary status are explicit session-manifest metadata.
- The processed dataset will be saved by trading session so individual sessions can be loaded efficiently without reading the complete history.

### Persisted V2 1-Second Fields

**Identity and time**
- UTC second timestamp
- CME session date
- instrument ID
- contract

**Price**
- open
- high
- low
- close

**Volume and trade activity**
- total volume
- trade count
- price × volume sum
- maximum trade size
- sum of squared trade sizes
- uptick count
- downtick count
- same-price trade count

**Live-compatible order flow**
- inferred buy volume
- inferred sell volume
- inferred delta

**Historical benchmark only**
- native buy volume
- native sell volume
- native delta

**Data integrity**
- first trade timestamp
- last trade timestamp
- contract-roll flag

VWAP and average trade size are derivable from the persisted additive fields; they are not stored V2 columns. The historical test material below led to the authoritative source-module implementation.

## Prototype 1-Second Tape Layer

Before processing the complete historical dataset, the permanent 1-second tape schema will be tested on the existing 100,000-trade sample.

The prototype will verify that:

- trades are assigned to the correct UTC second and CME session date
- OHLC prices are aggregated correctly
- volume and trade activity are preserved
- `tick_rule_v1` order-flow information is preserved
- Databento native aggressor information remains available only as a historical benchmark
- the resulting fields are sufficient for later 1-minute and rolling feature engineering

Once the prototype has been inspected and validated, the same design can be adapted to memory-safe chunked processing of the complete historical dataset.

In [20]:
# Build a prototype of the permanent 1-second tape layer from the
# 100,000-trade sample already loaded in trades_work.

prototype = trades_work.copy()

# Use UTC as the canonical stored timestamp. The local Chicago timestamp
# will only be used to determine the CME trading-session date.
prototype["timestamp_utc"] = prototype["ts_event"].dt.tz_convert("UTC")
prototype["timestamp_second"] = prototype["timestamp_utc"].dt.floor("s")

# Derive Chicago time for CME session assignment.
chicago_time = prototype["timestamp_utc"].dt.tz_convert("America/Chicago")

# CME equity-index futures trade date:
# trades at or after 5:00 PM CT belong to the following session date.
prototype["session_date"] = chicago_time.dt.date
after_session_open = chicago_time.dt.hour >= 17
prototype.loc[after_session_open, "session_date"] = (
    chicago_time.loc[after_session_open] + pd.Timedelta(days=1)
).dt.date

# Convert Databento's native aggressor side into signed direction.
# This is retained only as a historical benchmark.
prototype["native_direction"] = (
    prototype["side"]
    .map({"B": 1, "A": -1})
    .fillna(0)
    .astype("int8")
)

prototype["native_buy_volume"] = prototype["size"].where(
    prototype["native_direction"] == 1, 0
)
prototype["native_sell_volume"] = prototype["size"].where(
    prototype["native_direction"] == -1, 0
)
prototype["native_signed_volume"] = (
    prototype["size"] * prototype["native_direction"]
)

# Use the already-created tick_rule_v1 direction for the live-compatible
# order-flow measurements.
prototype["inferred_buy_volume"] = prototype["size"].where(
    prototype["inferred_direction"] == 1, 0
)
prototype["inferred_sell_volume"] = prototype["size"].where(
    prototype["inferred_direction"] == -1, 0
)
prototype["inferred_signed_volume"] = (
    prototype["size"] * prototype["inferred_direction"]
)

# Preserve additive quantities that allow useful statistics such as VWAP
# and trade-size dispersion to be calculated later from the 1-second layer.
prototype["price_volume"] = prototype["price"] * prototype["size"]
prototype["size_squared"] = prototype["size"] ** 2

# Classify price behavior for tape-activity counts.
price_change = prototype["price"].diff()
prototype["uptick"] = (price_change > 0).astype("int8")
prototype["downtick"] = (price_change < 0).astype("int8")
prototype["same_price"] = (price_change == 0).astype("int8")

# Aggregate individual trades into the prototype permanent 1-second layer.
second_prototype = (
    prototype
    .groupby(
        ["timestamp_second", "session_date", "instrument_id"],
        as_index=False
    )
    .agg(
        open=("price", "first"),
        high=("price", "max"),
        low=("price", "min"),
        close=("price", "last"),
        total_volume=("size", "sum"),
        trade_count=("size", "size"),
        price_volume_sum=("price_volume", "sum"),
        max_trade_size=("size", "max"),
        size_squared_sum=("size_squared", "sum"),
        uptick_count=("uptick", "sum"),
        downtick_count=("downtick", "sum"),
        same_price_count=("same_price", "sum"),
        inferred_buy_volume=("inferred_buy_volume", "sum"),
        inferred_sell_volume=("inferred_sell_volume", "sum"),
        inferred_delta=("inferred_signed_volume", "sum"),
        native_buy_volume=("native_buy_volume", "sum"),
        native_sell_volume=("native_sell_volume", "sum"),
        native_delta=("native_signed_volume", "sum"),
        first_trade_timestamp=("timestamp_utc", "first"),
        last_trade_timestamp=("timestamp_utc", "last"),
    )
)

# VWAP is included for inspection, although it can always be reconstructed
# from price_volume_sum / total_volume later.
second_prototype["vwap"] = (
    second_prototype["price_volume_sum"]
    / second_prototype["total_volume"]
)

print(f"Raw sample trades: {len(prototype):,}")
print(f"Prototype 1-second rows: {len(second_prototype):,}")
print(f"Columns: {len(second_prototype.columns)}")

second_prototype.head(10)

Raw sample trades: 100,000
Prototype 1-second rows: 20,682
Columns: 24


## Permanent v2 1-Second Foundation Schema

The event-time v2 dataset has exactly 33 columns. Its physical Arrow schema is
also fixed: prices and price-weighted sums are `float64`; non-negative volumes
and squared-size sums are `uint64`; counts are `uint32`; deltas are `int64`;
timestamps are UTC nanoseconds; and the CME session label is a date.

This is deliberate. Volumes are widened before squaring so a `uint32` trade
size cannot overflow during `size ** 2`. Delta validation casts components to
signed integers before subtraction, because unsigned buy-minus-sell arithmetic
can silently wrap. Stable types ensure every session has the same Parquet
schema, regardless of whether an unusually active second occurs that day.

`uptick_count`, `downtick_count`, and `same_price_count` count only trades with
a valid previous price in the same contract. The initial trade and the first
trade after a contract reset are not falsely labelled same-price; consequently,
their three-way sum may be less than `trade_count` by the number of resets.

## One-Session Production Test

This historical one-session test preceded the completed full-history V2 production run.

This test will verify that:

- raw trades can be processed in memory-safe chunks
- `tick_rule_v1` correctly carries state across chunk boundaries
- 1-second rows split across chunks are consolidated correctly
- CME session dates are assigned correctly
- contract identity is preserved
- all permanent schema fields are created
- inferred and native volume totals reconcile
- duplicate second/instrument rows are eliminated
- the final Parquet file can be saved and reloaded successfully

The current production path is `src/trade_processing_v2.py`; this section remains as learning and validation history.

In [21]:
# Define the one-session raw DBN file and the test output location.
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

test_raw_path = project_root / "data" / "mes_trades_2026-09-08_session.dbn"
test_output_path = (
    project_root
    / "data"
    / "processed"
    / "1s"
    / "MES_1s_2026-09-08_test.parquet"
)

print("Raw test file:")
print(test_raw_path)

print("\nTest output file:")
print(test_output_path)

print("\nRaw file exists:", test_raw_path.exists())
print("Output directory exists:", test_output_path.parent.exists())

Raw test file:
/Users/marcusromeo/Desktop/mes-trading-project/data/mes_trades_2026-09-08_session.dbn

Test output file:
/Users/marcusromeo/Desktop/mes-trading-project/data/processed/1s/MES_1s_2026-09-08_test.parquet

Raw file exists: True
Output directory exists: True


In [22]:
# Map Databento instrument IDs to the actual MES contracts identified
# during raw-data validation. Keeping both fields lets us detect contract
# changes and prevents futures-roll price jumps from becoming model returns.

contract_map = {
    42004164: "MESZ5",
    42003800: "MESH6",
    42005163: "MESM6",
    42003239: "MESU6",
}

print("MES contract mapping:")
for instrument_id, contract in contract_map.items():
    print(f"{instrument_id} -> {contract}")

MES contract mapping:
42004164 -> MESZ5
42003800 -> MESH6
42005163 -> MESM6
42003239 -> MESU6


In [23]:
# Use the one authoritative tick-rule implementation in every later test.
# It is live-compatible: price up/down establishes direction, unchanged prices
# inherit the latest direction, and a futures-contract change resets state.
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from trade_processing_v2 import apply_tick_rule_v1

print('Authoritative tick_rule_v1 imported.')

In [24]:
# Test tick_rule_v1 on a tiny artificial trade sequence, including
# same-price trades and a contract change that should reset the rule.

test_prices = [100.00, 100.25, 100.25, 100.00, 100.00, 101.00, 101.25]
test_instruments = [1, 1, 1, 1, 1, 2, 2]

test_directions, final_price, final_direction, final_instrument = apply_tick_rule_v1(
    test_prices,
    test_instruments,
)

print("Prices:     ", test_prices)
print("Instruments:", test_instruments)
print("Directions:", test_directions.tolist())
print()
print("Expected:   ", [0, 1, 1, -1, -1, 0, 1])
print("Test passed:", test_directions.tolist() == [0, 1, 1, -1, -1, 0, 1])

Prices:      [100.0, 100.25, 100.25, 100.0, 100.0, 101.0, 101.25]
Instruments: [1, 1, 1, 1, 1, 2, 2]
Directions: [0, 1, 1, -1, -1, 0, 1]

Expected:    [0, 1, 1, -1, -1, 0, 1]
Test passed: True


In [25]:
# Verify that tick_rule_v1 produces the same result when the same trade
# sequence is processed in multiple chunks instead of all at once.

full_prices = [100.00, 100.25, 100.25, 100.00, 100.00, 100.25, 100.25]
full_instruments = [1, 1, 1, 1, 1, 1, 1]

# Process the entire sequence at once.
full_directions, _, _, _ = apply_tick_rule_v1(
    full_prices,
    full_instruments,
)

# Process the same sequence in two chunks.
chunk_1_prices = full_prices[:4]
chunk_1_instruments = full_instruments[:4]

chunk_1_directions, prev_price, prev_direction, prev_instrument = apply_tick_rule_v1(
    chunk_1_prices,
    chunk_1_instruments,
)

chunk_2_prices = full_prices[4:]
chunk_2_instruments = full_instruments[4:]

chunk_2_directions, _, _, _ = apply_tick_rule_v1(
    chunk_2_prices,
    chunk_2_instruments,
    previous_price=prev_price,
    previous_direction=prev_direction,
    previous_instrument_id=prev_instrument,
)

chunked_directions = (
    chunk_1_directions.tolist()
    + chunk_2_directions.tolist()
)

print("Full-pass directions: ", full_directions.tolist())
print("Chunked directions:   ", chunked_directions)
print()
print(
    "Chunk-boundary test passed:",
    full_directions.tolist() == chunked_directions
)

Full-pass directions:  [0, 1, 1, -1, -1, 1, 1]
Chunked directions:    [0, 1, 1, -1, -1, 1, 1]

Chunk-boundary test passed: True


In [26]:
# Open the one-session Databento file as a chunked DataFrame iterator.
# We use chunks so the same approach will scale later to the full 99M-trade file.

from databento import DBNStore

test_store = DBNStore.from_file(test_raw_path)

test_chunks = test_store.to_df(
    count=500_000,
)

first_chunk = next(iter(test_chunks))

print("First chunk loaded.")
print(f"Rows: {len(first_chunk):,}")
print()
print("Columns:")
print(first_chunk.columns.tolist())
print()
print("First timestamp:")
print(first_chunk["ts_event"].iloc[0])
print()
print("Last timestamp:")
print(first_chunk["ts_event"].iloc[-1])
print()
print("Instrument IDs:")
print(first_chunk["instrument_id"].unique())

First chunk loaded.
Rows: 296,943

Columns:
['ts_event', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side', 'depth', 'price', 'size', 'flags', 'ts_in_delta', 'sequence', 'symbol']

First timestamp:
2026-09-08 22:00:00+00:00

Last timestamp:
2026-09-09 20:59:59.523350363+00:00

Instrument IDs:
[42003239]


In [27]:
# Transform the one-session real MES trade data into the permanent
# 1-second foundation schema before saving anything to disk.

session_trades = first_chunk.copy()

# Create canonical UTC timestamps and assign each trade to its UTC second.
session_trades["timestamp_utc"] = session_trades["ts_event"].dt.tz_convert("UTC")
session_trades["timestamp_second"] = session_trades["timestamp_utc"].dt.floor("s")

# Derive the CME trading-session date using Chicago time.
chicago_time = session_trades["timestamp_utc"].dt.tz_convert("America/Chicago")
session_trades["session_date"] = chicago_time.dt.date

after_session_open = chicago_time.dt.hour >= 17
session_trades.loc[after_session_open, "session_date"] = (
    chicago_time.loc[after_session_open] + pd.Timedelta(days=1)
).dt.date

# Map the Databento instrument ID to the actual MES futures contract.
session_trades["contract"] = session_trades["instrument_id"].map(contract_map)

# Apply the validated live-compatible tick_rule_v1 classifier.
session_trades["inferred_direction"], _, _, _ = apply_tick_rule_v1(
    session_trades["price"].to_numpy(),
    session_trades["instrument_id"].to_numpy(),
)

# Convert Databento's historical native aggressor side into signed direction.
session_trades["native_direction"] = (
    session_trades["side"]
    .map({"B": 1, "A": -1})
    .fillna(0)
    .astype("int8")
)

print("Real one-session trade preparation complete.")
print(f"Trades prepared: {len(session_trades):,}")
print("Session dates:", session_trades["session_date"].unique())
print("Contracts:", session_trades["contract"].unique())
print("Unmapped contracts:", session_trades["contract"].isna().sum())

Real one-session trade preparation complete.
Trades prepared: 296,943
Session dates: [datetime.date(2026, 9, 9)]
Contracts: <ArrowStringArray>
['MESU6']
Length: 1, dtype: str
Unmapped contracts: 0


In [28]:
# Build the permanent 1-second foundation rows from the prepared real MES trades.

# Create trade-level ingredients needed for later 1-second aggregation.
session_trades["price_volume"] = (
    session_trades["price"] * session_trades["size"]
)

session_trades["price_squared_volume"] = (
    (session_trades["price"] ** 2) * session_trades["size"]
)

session_trades["size_squared"] = (
    session_trades["size"] ** 2
)

# Create inferred buy/sell/unknown volume and trade-count fields.
session_trades["inferred_buy_volume"] = session_trades["size"].where(
    session_trades["inferred_direction"] == 1, 0
)

session_trades["inferred_sell_volume"] = session_trades["size"].where(
    session_trades["inferred_direction"] == -1, 0
)

session_trades["inferred_unknown_volume"] = session_trades["size"].where(
    session_trades["inferred_direction"] == 0, 0
)

session_trades["inferred_buy_trade"] = (
    session_trades["inferred_direction"] == 1
).astype("int8")

session_trades["inferred_sell_trade"] = (
    session_trades["inferred_direction"] == -1
).astype("int8")

session_trades["inferred_unknown_trade"] = (
    session_trades["inferred_direction"] == 0
).astype("int8")

session_trades["inferred_signed_volume"] = (
    session_trades["size"] * session_trades["inferred_direction"]
)

# Create Databento-native benchmark volume fields.
session_trades["native_buy_volume"] = session_trades["size"].where(
    session_trades["native_direction"] == 1, 0
)

session_trades["native_sell_volume"] = session_trades["size"].where(
    session_trades["native_direction"] == -1, 0
)

session_trades["native_unknown_volume"] = session_trades["size"].where(
    session_trades["native_direction"] == 0, 0
)

session_trades["native_signed_volume"] = (
    session_trades["size"] * session_trades["native_direction"]
)

# Classify each trade's price movement relative to the previous trade.
price_change = session_trades["price"].diff()

session_trades["uptick"] = (price_change > 0).astype("int8")
session_trades["downtick"] = (price_change < 0).astype("int8")
session_trades["same_price"] = (price_change == 0).astype("int8")

print("Trade-level aggregation ingredients created.")
print(f"Prepared trades: {len(session_trades):,}")

Trade-level aggregation ingredients created.
Prepared trades: 296,943


In [29]:
# Aggregate the prepared MES trades into the permanent 1-second foundation schema.

second_data = (
    session_trades
    .groupby(
        ["timestamp_second", "session_date", "instrument_id", "contract"],
        as_index=False
    )
    .agg(
        # Price information.
        open=("price", "first"),
        high=("price", "max"),
        low=("price", "min"),
        close=("price", "last"),
        price_volume_sum=("price_volume", "sum"),
        price_squared_volume_sum=("price_squared_volume", "sum"),

        # Trading activity.
        total_volume=("size", "sum"),
        trade_count=("size", "size"),
        max_trade_size=("size", "max"),
        size_squared_sum=("size_squared", "sum"),

        # Price / tape behavior.
        uptick_count=("uptick", "sum"),
        downtick_count=("downtick", "sum"),
        same_price_count=("same_price", "sum"),
        unique_price_levels=("price", "nunique"),

        # Live-compatible inferred order flow.
        inferred_buy_volume=("inferred_buy_volume", "sum"),
        inferred_sell_volume=("inferred_sell_volume", "sum"),
        inferred_unknown_volume=("inferred_unknown_volume", "sum"),
        inferred_buy_trade_count=("inferred_buy_trade", "sum"),
        inferred_sell_trade_count=("inferred_sell_trade", "sum"),
        inferred_unknown_trade_count=("inferred_unknown_trade", "sum"),
        inferred_delta=("inferred_signed_volume", "sum"),

        # Databento-native historical benchmark.
        native_buy_volume=("native_buy_volume", "sum"),
        native_sell_volume=("native_sell_volume", "sum"),
        native_unknown_volume=("native_unknown_volume", "sum"),
        native_delta=("native_signed_volume", "sum"),

        # Preserve exact timing information for integrity checks.
        first_trade_timestamp=("timestamp_utc", "first"),
        last_trade_timestamp=("timestamp_utc", "last"),
    )
)

# Calculate the largest amount of volume traded at any single price
# during each second. This preserves concentration information that
# cannot be reconstructed from ordinary OHLCV data later.
volume_at_price = (
    session_trades
    .groupby(
        ["timestamp_second", "session_date", "instrument_id", "contract", "price"],
        as_index=False
    )["size"]
    .sum()
)

max_volume_at_price = (
    volume_at_price
    .groupby(
        ["timestamp_second", "session_date", "instrument_id", "contract"],
        as_index=False
    )["size"]
    .max()
    .rename(columns={"size": "max_volume_at_price"})
)

second_data = second_data.merge(
    max_volume_at_price,
    on=["timestamp_second", "session_date", "instrument_id", "contract"],
    how="left"
)

# This one-session test contains only one contract, so there should
# be no contract-change second inside this file.
second_data["contract_change"] = (
    second_data["instrument_id"]
    != second_data["instrument_id"].shift(1)
)

# The first row has no previous row inside this test file, so it is
# not treated as an observed contract change.
second_data.loc[second_data.index[0], "contract_change"] = False

print("1-second foundation layer created.")
print(f"Raw trades: {len(session_trades):,}")
print(f"1-second rows: {len(second_data):,}")
print(f"Columns: {len(second_data.columns)}")
print()
print("Session dates:", second_data["session_date"].unique())
print("Contracts:", second_data["contract"].unique())
print("Contract changes:", second_data["contract_change"].sum())

second_data.head(10)

1-second foundation layer created.
Raw trades: 296,943
1-second rows: 51,535
Columns: 33

Session dates: [datetime.date(2026, 9, 9)]
Contracts: <ArrowStringArray>
['MESU6']
Length: 1, dtype: str
Contract changes: 0


In [33]:
# Validate the completed 1-second foundation layer before saving it.
# These checks confirm that timestamps, OHLC values, volume, delta,
# and trade-level summary fields are internally consistent.

# Calculate expected delta using signed integers.
# The volume columns are uint32, so converting to int64 prevents
# unsigned arithmetic from wrapping when sell volume exceeds buy volume.
inferred_expected_delta = (
    second_data["inferred_buy_volume"].astype("int64")
    - second_data["inferred_sell_volume"].astype("int64")
)

native_expected_delta = (
    second_data["native_buy_volume"].astype("int64")
    - second_data["native_sell_volume"].astype("int64")
)

checks = {
    # Each timestamp/instrument combination should appear only once.
    "unique second rows": ~second_data.duplicated(
        subset=["timestamp_second", "instrument_id"]
    ).any(),

    # The data should remain in chronological order.
    "timestamps sorted": second_data["timestamp_second"].is_monotonic_increasing,

    # OHLC prices must be internally consistent.
    "high >= open": (second_data["high"] >= second_data["open"]).all(),
    "high >= close": (second_data["high"] >= second_data["close"]).all(),
    "low <= open": (second_data["low"] <= second_data["open"]).all(),
    "low <= close": (second_data["low"] <= second_data["close"]).all(),

    # Every recorded second must contain trades and positive volume.
    "positive volume": (second_data["total_volume"] > 0).all(),
    "positive trade count": (second_data["trade_count"] > 0).all(),

    # Buy + sell + unknown volume must reconstruct total volume.
    "inferred volume reconciles": (
        second_data["inferred_buy_volume"].astype("int64")
        + second_data["inferred_sell_volume"].astype("int64")
        + second_data["inferred_unknown_volume"].astype("int64")
        == second_data["total_volume"].astype("int64")
    ).all(),

    "native volume reconciles": (
        second_data["native_buy_volume"].astype("int64")
        + second_data["native_sell_volume"].astype("int64")
        + second_data["native_unknown_volume"].astype("int64")
        == second_data["total_volume"].astype("int64")
    ).all(),

    # Delta must equal buy volume minus sell volume.
    "inferred delta reconciles": (
        second_data["inferred_delta"].astype("int64")
        == inferred_expected_delta
    ).all(),

    "native delta reconciles": (
        second_data["native_delta"].astype("int64")
        == native_expected_delta
    ).all(),

    # Maximum volume at a single price cannot exceed total second volume.
    "max volume at price valid": (
        second_data["max_volume_at_price"] <= second_data["total_volume"]
    ).all(),

    # First trade must occur before or at the last trade in each second.
    "trade timestamps ordered": (
        second_data["first_trade_timestamp"]
        <= second_data["last_trade_timestamp"]
    ).all(),
}

print("ONE-SECOND FOUNDATION VALIDATION")
print("--------------------------------")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print()
print(f"All checks passed: {all(checks.values())}")

ONE-SECOND FOUNDATION VALIDATION
--------------------------------
unique second rows: PASS
timestamps sorted: PASS
high >= open: PASS
high >= close: PASS
low <= open: PASS
low <= close: PASS
positive volume: PASS
positive trade count: PASS
inferred volume reconciles: PASS
native volume reconciles: PASS
inferred delta reconciles: PASS
native delta reconciles: PASS
max volume at price valid: PASS
trade timestamps ordered: PASS

All checks passed: True


## Parquet Storage Round-Trip Test

This historical test validated a one-session storage round trip before the completed full-history V2 run. It is retained for learning context, but is disabled so notebook execution does not create persistent files.

The authoritative writer now lives in `src/trade_processing_v2.py`; it enforces the fixed Arrow schema and refuses to overwrite an existing session file.

In [34]:
# Historical direct-write example intentionally disabled.
# The authoritative writer in trade_processing_v2.py validates the fixed
# schema and refuses to overwrite an existing session. Keeping this cell
# non-writing makes "Run All" safe for the repository data directories.
RUN_HISTORICAL_PARQUET_WRITE_TEST = False

print("Historical persistent Parquet write skipped.")

In [35]:
# This companion reload check is disabled with the historical writer above.
# The completed V2 audit performs the authoritative on-disk schema and
# session validation without relying on this prototype path.

print("Historical persistent Parquet reload test skipped.")

## Full-History Chunked Processor

The historical one-session transformation and storage format were validated before the completed V2 run.

The full historical dataset contains approximately 99.3 million individual MES trades and cannot be loaded into memory at once. The production processor therefore reads the raw DBN file in manageable chunks while preserving state across chunk boundaries.

The authoritative processor produces the permanent 1-second schema while correctly handling:

- tick-rule state across chunks
- seconds split between adjacent chunks
- CME trading-session boundaries
- futures contract changes
- per-session Parquet output
- memory usage suitable for the local machine

The completed V2 dataset contains 242 session files. Feature engineering should use that approved 1-second layer rather than repeatedly scanning the raw 99-million-trade DBN file.

In [36]:
# Import the authoritative event-time context helper.
# It derives all market timestamps from ts_event, never the ts_recv DataFrame index.
from trade_processing_v2 import CONTRACT_MAP, add_trade_context, assert_event_time_order

print('Event-time context and chronological-order helpers imported.')

In [37]:
# Import the validated incomplete-second helper.
# The final UTC event-time second stays pending until the next raw chunk proves
# it is complete; this prevents one bar from being split across chunks.
from trade_processing_v2 import split_complete_seconds

print('Incomplete-second carryover helper imported.')

In [38]:
# Test that split_complete_seconds correctly holds back the final second
# so that trades from the same second are never split across chunks.

test_times = pd.to_datetime(
    [
        "2026-09-09 14:32:04.100000+00:00",
        "2026-09-09 14:32:04.500000+00:00",
        "2026-09-09 14:32:05.100000+00:00",
        "2026-09-09 14:32:05.300000+00:00",
        "2026-09-09 14:32:05.600000+00:00",
    ]
)

test_trades = pd.DataFrame(
    {
        "timestamp": test_times,
        "timestamp_second": test_times.floor("s"),
        "price": [7680.00, 7680.25, 7680.50, 7680.25, 7680.50],
    }
)

complete_test, pending_test = split_complete_seconds(test_trades)

print("Complete rows:")
print(complete_test[["timestamp", "timestamp_second", "price"]])

print()
print("Pending rows:")
print(pending_test[["timestamp", "timestamp_second", "price"]])

print()
print(f"Complete rows: {len(complete_test)}")
print(f"Pending rows: {len(pending_test)}")

Complete rows:
                         timestamp          timestamp_second    price
0 2026-09-09 14:32:04.100000+00:00 2026-09-09 14:32:04+00:00  7680.00
1 2026-09-09 14:32:04.500000+00:00 2026-09-09 14:32:04+00:00  7680.25

Pending rows:
                         timestamp          timestamp_second    price
2 2026-09-09 14:32:05.100000+00:00 2026-09-09 14:32:05+00:00  7680.50
3 2026-09-09 14:32:05.300000+00:00 2026-09-09 14:32:05+00:00  7680.25
4 2026-09-09 14:32:05.600000+00:00 2026-09-09 14:32:05+00:00  7680.50

Complete rows: 2
Pending rows: 3


In [39]:
# Simulate the next raw chunk and prepend the pending trades from chunk 1.
# This verifies that a second split across two chunks is reconstructed before processing.

next_times = pd.to_datetime(
    [
        "2026-09-09 14:32:05.700000+00:00",
        "2026-09-09 14:32:05.900000+00:00",
        "2026-09-09 14:32:06.100000+00:00",
        "2026-09-09 14:32:06.400000+00:00",
    ]
)

next_chunk = pd.DataFrame(
    {
        "timestamp": next_times,
        "timestamp_second": next_times.floor("s"),
        "price": [7680.75, 7681.00, 7681.25, 7681.00],
    }
)

# Reattach the pending trades from the previous chunk.
combined_chunk = pd.concat(
    [pending_test, next_chunk],
    ignore_index=True,
)

# Again hold back only the final second of the newly combined chunk.
complete_test_2, pending_test_2 = split_complete_seconds(combined_chunk)

print("Complete rows after combining chunks:")
print(complete_test_2[["timestamp", "timestamp_second", "price"]])

print()
print("New pending rows:")
print(pending_test_2[["timestamp", "timestamp_second", "price"]])

print()
print(f"Complete rows: {len(complete_test_2)}")
print(f"Pending rows: {len(pending_test_2)}")

# Confirm that all five trades from 14:32:05 are now together.
reconstructed_second = complete_test_2[
    complete_test_2["timestamp_second"]
    == pd.Timestamp("2026-09-09 14:32:05+00:00")
]

print()
print(f"Trades reconstructed for 14:32:05: {len(reconstructed_second)}")
print(f"Boundary reconstruction passed: {len(reconstructed_second) == 5}")

Complete rows after combining chunks:
                         timestamp          timestamp_second    price
0 2026-09-09 14:32:05.100000+00:00 2026-09-09 14:32:05+00:00  7680.50
1 2026-09-09 14:32:05.300000+00:00 2026-09-09 14:32:05+00:00  7680.25
2 2026-09-09 14:32:05.600000+00:00 2026-09-09 14:32:05+00:00  7680.50
3 2026-09-09 14:32:05.700000+00:00 2026-09-09 14:32:05+00:00  7680.75
4 2026-09-09 14:32:05.900000+00:00 2026-09-09 14:32:05+00:00  7681.00

New pending rows:
                         timestamp          timestamp_second    price
5 2026-09-09 14:32:06.100000+00:00 2026-09-09 14:32:06+00:00  7681.25
6 2026-09-09 14:32:06.400000+00:00 2026-09-09 14:32:06+00:00  7681.00

Complete rows: 5
Pending rows: 2

Trades reconstructed for 14:32:05: 5
Boundary reconstruction passed: True


In [40]:
# Import the authoritative 1-second aggregation helper.
# It accepts complete, prepared event-time trades and returns the 32 base fields;
# contract_change is intentionally added afterward because it needs batch state.
from trade_processing_v2 import aggregate_to_one_second

print('Authoritative aggregation helper imported.')

In [41]:
# Test the reusable aggregation function against the already validated
# one-session foundation layer before using it in the full-history processor.

function_test = aggregate_to_one_second(session_trades)

# The original validated second_data contains contract_change as its 33rd column.
# The reusable aggregation function intentionally leaves that field out because
# contract changes must be detected using state across processing boundaries.
common_columns = [
    column
    for column in second_data.columns
    if column != "contract_change"
]

original_common = (
    second_data[common_columns]
    .reset_index(drop=True)
)

function_common = (
    function_test[common_columns]
    .reset_index(drop=True)
)

# Compare the actual values independently of minor pandas dtype differences.
pd.testing.assert_frame_equal(
    original_common,
    function_common,
    check_dtype=False,
)

print("REUSABLE AGGREGATION TEST")
print("-------------------------")
print(f"Original rows: {len(original_common):,}")
print(f"Function rows: {len(function_common):,}")
print(f"Columns compared: {len(common_columns)}")
print("All aggregation values match: True")

REUSABLE AGGREGATION TEST
-------------------------
Original rows: 51,535
Function rows: 51,535
Columns compared: 32
All aggregation values match: True


In [42]:
# Import contract-roll detection with cross-batch state.
# A roll is flagged on the first aggregated second of the new instrument.
from trade_processing_v2 import add_contract_change_flag

print('Authoritative contract-change helper imported.')

In [43]:
# Test that contract_change is detected correctly even when the futures roll
# happens between two separately processed 1-second batches.

test_batch_1 = pd.DataFrame(
    {
        "timestamp_second": pd.to_datetime(
            [
                "2025-12-17 14:00:00+00:00",
                "2025-12-17 14:00:01+00:00",
            ]
        ),
        "instrument_id": [42004164, 42004164],  # MESZ5
    }
)

test_batch_2 = pd.DataFrame(
    {
        "timestamp_second": pd.to_datetime(
            [
                "2025-12-17 14:00:02+00:00",
                "2025-12-17 14:00:03+00:00",
            ]
        ),
        "instrument_id": [42003800, 42003800],  # MESH6
    }
)

# Process the first batch.
flagged_1, previous_instrument_id = add_contract_change_flag(
    test_batch_1,
    previous_instrument_id=None,
)

# Carry the final instrument from batch 1 into batch 2.
flagged_2, previous_instrument_id = add_contract_change_flag(
    test_batch_2,
    previous_instrument_id=previous_instrument_id,
)

print("Batch 1:")
print(flagged_1[["timestamp_second", "instrument_id", "contract_change"]])

print()
print("Batch 2:")
print(flagged_2[["timestamp_second", "instrument_id", "contract_change"]])

print()

expected_batch_1 = [False, False]
expected_batch_2 = [True, False]

print(
    "Batch 1 correct:",
    flagged_1["contract_change"].tolist() == expected_batch_1,
)

print(
    "Batch 2 correct:",
    flagged_2["contract_change"].tolist() == expected_batch_2,
)

print(
    "Cross-boundary contract roll test passed:",
    (
        flagged_1["contract_change"].tolist() == expected_batch_1
        and flagged_2["contract_change"].tolist() == expected_batch_2
    ),
)

Batch 1:
           timestamp_second  instrument_id  contract_change
0 2025-12-17 14:00:00+00:00       42004164            False
1 2025-12-17 14:00:01+00:00       42004164            False

Batch 2:
           timestamp_second  instrument_id  contract_change
0 2025-12-17 14:00:02+00:00       42003800             True
1 2025-12-17 14:00:03+00:00       42003800            False

Batch 1 correct: True
Batch 2 correct: True
Cross-boundary contract roll test passed: True


In [55]:
# Import preparation logic that carries tick and price-change state.
# Its no-prior-price rule intentionally leaves initialization and contract-reset
# trades out of uptick/downtick/same-price counts.
from trade_processing_v2 import prepare_complete_trades

print('Authoritative stateful preparation helper imported.')

In [56]:
# Test that price-change classification remains correct across a processing boundary.
# Batch 2 begins above the final price of Batch 1, so its first trade must be an uptick.

price_test_1 = pd.DataFrame(
    {
        "timestamp": pd.to_datetime([
            "2026-09-09 14:32:04.100000+00:00",
            "2026-09-09 14:32:04.500000+00:00",
        ]),
        "instrument_id": [42003239, 42003239],
        "price": [7680.00, 7680.25],
        "size": [1, 1],
        "side": ["B", "B"],
    }
)

price_test_2 = pd.DataFrame(
    {
        "timestamp": pd.to_datetime([
            "2026-09-09 14:32:05.100000+00:00",
            "2026-09-09 14:32:05.500000+00:00",
        ]),
        "instrument_id": [42003239, 42003239],
        "price": [7680.50, 7680.50],
        "size": [1, 1],
        "side": ["B", "B"],
    }
)

# Start with no previous processing state.
previous_price = None
previous_direction = 0
previous_instrument_id = None

prepared_1, previous_price, previous_direction, previous_instrument_id = (
    prepare_complete_trades(
        price_test_1,
        previous_price,
        previous_direction,
        previous_instrument_id,
    )
)

prepared_2, previous_price, previous_direction, previous_instrument_id = (
    prepare_complete_trades(
        price_test_2,
        previous_price,
        previous_direction,
        previous_instrument_id,
    )
)

print("Batch 1:")
print(prepared_1[["price", "uptick", "downtick", "same_price"]])

print()
print("Batch 2:")
print(prepared_2[["price", "uptick", "downtick", "same_price"]])

# The first trade of Batch 2 is 7680.50 versus the previous batch's
# final price of 7680.25, so it must be classified as an uptick.
boundary_uptick_passed = prepared_2.iloc[0]["uptick"] == 1

# The second trade remains at 7680.50, so it must be same-price.
same_price_passed = prepared_2.iloc[1]["same_price"] == 1

print()
print(f"Boundary uptick correct: {boundary_uptick_passed}")
print(f"Following same-price correct: {same_price_passed}")
print(
    "Price-change boundary test passed:",
    boundary_uptick_passed and same_price_passed,
)

In [57]:
# Create a small real-data sample for testing the full chunk-processing workflow.
# We intentionally use small chunks so several artificial boundaries occur
# inside real MES data, letting us verify that state is preserved correctly.

integration_test_trades = session_trades.iloc[:50_000].copy()

test_chunk_size = 10_000

print("INTEGRATION TEST SETUP")
print("----------------------")
print(f"Real trades selected: {len(integration_test_trades):,}")
print(f"Artificial chunk size: {test_chunk_size:,}")
print(
    "Expected chunks:",
    (len(integration_test_trades) + test_chunk_size - 1) // test_chunk_size,
)

# Show the exact UTC timestamp range covered by this real-data integration sample.
print(
    "First timestamp:",
    integration_test_trades["timestamp_utc"].iloc[0],
)
print(
    "Last timestamp:",
    integration_test_trades["timestamp_utc"].iloc[-1],
)

INTEGRATION TEST SETUP
----------------------
Real trades selected: 50,000
Artificial chunk size: 10,000
Expected chunks: 5
First timestamp: 2026-09-08 22:00:00+00:00
Last timestamp: 2026-09-09 10:42:08.499172555+00:00


In [59]:
# Integration test: compare five artificial chunks against one uninterrupted pass
# over the exact same 50,000 real MES trades.
#
# The one-session dataset stores its timestamp as timestamp_utc, while the
# production preparation function expects the raw-style column name timestamp.
# Create that alias only inside this test copy.

integration_test_input = integration_test_trades.copy()
integration_test_input["timestamp"] = integration_test_input["timestamp_utc"]

# ------------------------------------------------------------
# 1. Process all 50,000 trades in one uninterrupted pass.
# ------------------------------------------------------------

continuous_prepared, _, _, _ = prepare_complete_trades(
    integration_test_input.copy(),
    previous_price=None,
    previous_direction=0,
    previous_instrument_id=None,
)

# ------------------------------------------------------------
# 2. Process the same trades as five separate 10,000-row chunks.
# ------------------------------------------------------------

chunked_results = []

previous_price = None
previous_direction = 0
previous_instrument_id = None

for start in range(0, len(integration_test_input), test_chunk_size):
    stop = min(
        start + test_chunk_size,
        len(integration_test_input),
    )

    chunk = integration_test_input.iloc[start:stop].copy()

    (
        prepared_chunk,
        previous_price,
        previous_direction,
        previous_instrument_id,
    ) = prepare_complete_trades(
        chunk,
        previous_price=previous_price,
        previous_direction=previous_direction,
        previous_instrument_id=previous_instrument_id,
    )

    chunked_results.append(prepared_chunk)

# Reassemble the separately processed chunks.
chunked_prepared = pd.concat(
    chunked_results,
    ignore_index=True,
)

continuous_prepared = continuous_prepared.reset_index(drop=True)

# ------------------------------------------------------------
# 3. Compare every state-dependent field.
# ------------------------------------------------------------

comparison_columns = [
    "inferred_direction",
    "uptick",
    "downtick",
    "same_price",
    "inferred_buy_volume",
    "inferred_sell_volume",
    "inferred_unknown_volume",
    "inferred_signed_volume",
]

column_results = {}

for column in comparison_columns:
    column_results[column] = (
        continuous_prepared[column].to_numpy()
        == chunked_prepared[column].to_numpy()
    ).all()

row_count_matches = (
    len(continuous_prepared)
    == len(chunked_prepared)
)

integration_test_passed = (
    row_count_matches
    and all(column_results.values())
)

# ------------------------------------------------------------
# 4. Display the results.
# ------------------------------------------------------------

print("CHUNK-PROCESSING INTEGRATION TEST")
print("---------------------------------")
print(f"Continuous rows: {len(continuous_prepared):,}")
print(f"Chunked rows:    {len(chunked_prepared):,}")
print(f"Row count matches: {row_count_matches}")
print()

for column, passed in column_results.items():
    print(f"{column}: {'PASS' if passed else 'FAIL'}")

print()
print("Integration test passed:", integration_test_passed)

CHUNK-PROCESSING INTEGRATION TEST
---------------------------------
Continuous rows: 50,000
Chunked rows:    50,000
Row count matches: True

inferred_direction: PASS
uptick: PASS
downtick: PASS
same_price: PASS
inferred_buy_volume: PASS
inferred_sell_volume: PASS
inferred_unknown_volume: PASS
inferred_signed_volume: PASS

Integration test passed: True


In [60]:
# Full real-data boundary test:
# Process the same 50,000 MES trades continuously and through five artificial
# chunks while holding back incomplete final seconds. The resulting 1-second
# foundation data should be identical.

# ------------------------------------------------------------
# 1. Build the uninterrupted reference result.
# ------------------------------------------------------------

reference_prepared, _, _, _ = prepare_complete_trades(
    integration_test_input.copy(),
    previous_price=None,
    previous_direction=0,
    previous_instrument_id=None,
)

reference_seconds = (
    aggregate_to_one_second(reference_prepared)
    .sort_values(["timestamp_second", "instrument_id"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Process the same trades through artificial chunks while
#    carrying the unfinished final second into the next chunk.
# ------------------------------------------------------------

pending_trades = pd.DataFrame()
chunk_second_results = []

previous_price = None
previous_direction = 0
previous_instrument_id = None

for start in range(0, len(integration_test_input), test_chunk_size):
    stop = min(
        start + test_chunk_size,
        len(integration_test_input),
    )

    raw_chunk = integration_test_input.iloc[start:stop].copy()

    # Reattach the unfinished final second from the previous chunk.
    if not pending_trades.empty:
        raw_chunk = pd.concat(
            [pending_trades, raw_chunk],
            ignore_index=True,
        )

    # Hold back the new final second so it cannot be aggregated prematurely.
    complete_trades, pending_trades = split_complete_seconds(raw_chunk)

    # Process only trades belonging to complete seconds.
    if not complete_trades.empty:
        (
            prepared_chunk,
            previous_price,
            previous_direction,
            previous_instrument_id,
        ) = prepare_complete_trades(
            complete_trades,
            previous_price=previous_price,
            previous_direction=previous_direction,
            previous_instrument_id=previous_instrument_id,
        )

        second_chunk = aggregate_to_one_second(prepared_chunk)
        chunk_second_results.append(second_chunk)

# ------------------------------------------------------------
# 3. At end-of-file, the final pending second is now known to be
#    complete relative to the available dataset, so process it.
# ------------------------------------------------------------

if not pending_trades.empty:
    (
        prepared_final,
        previous_price,
        previous_direction,
        previous_instrument_id,
    ) = prepare_complete_trades(
        pending_trades,
        previous_price=previous_price,
        previous_direction=previous_direction,
        previous_instrument_id=previous_instrument_id,
    )

    final_seconds = aggregate_to_one_second(prepared_final)
    chunk_second_results.append(final_seconds)

# Combine all emitted 1-second rows.
chunked_seconds = (
    pd.concat(
        chunk_second_results,
        ignore_index=True,
    )
    .sort_values(["timestamp_second", "instrument_id"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Compare the entire 32-column aggregation against the
#    uninterrupted reference result.
# ------------------------------------------------------------

row_count_matches = len(reference_seconds) == len(chunked_seconds)
column_names_match = (
    reference_seconds.columns.tolist()
    == chunked_seconds.columns.tolist()
)

# This compares every stored value while allowing harmless dtype differences.
pd.testing.assert_frame_equal(
    reference_seconds,
    chunked_seconds,
    check_dtype=False,
)

duplicate_seconds = chunked_seconds.duplicated(
    subset=["timestamp_second", "instrument_id"]
).sum()

print("PARTIAL-SECOND INTEGRATION TEST")
print("--------------------------------")
print(f"Reference 1-second rows: {len(reference_seconds):,}")
print(f"Chunked 1-second rows:   {len(chunked_seconds):,}")
print(f"Row count matches: {row_count_matches}")
print(f"Column names match: {column_names_match}")
print(f"Duplicate second rows: {duplicate_seconds:,}")
print("All 1-second aggregation values match: True")
print()
print(
    "Partial-second integration test passed:",
    (
        row_count_matches
        and column_names_match
        and duplicate_seconds == 0
    ),
)

PARTIAL-SECOND INTEGRATION TEST
--------------------------------
Reference 1-second rows: 20,529
Chunked 1-second rows:   20,529
Row count matches: True
Column names match: True
Duplicate second rows: 0
All 1-second aggregation values match: True

Partial-second integration test passed: True


In [61]:
# Historical per-session writer test using the validated real-data
# integration result. It is disabled by default so "Run All" cannot
# create or overwrite persistent test artifacts.
#
# This historical test documented the storage side of the pipeline:
# 1-second rows -> CME session -> Parquet file -> reload -> exact comparison.

from pathlib import Path

# ------------------------------------------------------------
# 1. Add the final contract_change field to create the complete
#    permanent 33-column foundation schema.
# ------------------------------------------------------------

session_writer_data, _ = add_contract_change_flag(
    reference_seconds.copy(),
    previous_instrument_id=None,
)

print("SESSION WRITER TEST")
print("-------------------")
print(f"Input 1-second rows: {len(session_writer_data):,}")
print(f"Columns: {len(session_writer_data.columns)}")
print(
    "Session dates:",
    sorted(session_writer_data["session_date"].unique()),
)
print()

# ------------------------------------------------------------
# 2. Keep the historical persistent write disabled by default.
# ------------------------------------------------------------

RUN_HISTORICAL_SESSION_WRITER_TEST = False
session_test_dir = Path(
    "../data/processed/1s/session_writer_test"
)

written_files = []
session_groups = (
    session_writer_data.groupby("session_date", sort=True)
    if RUN_HISTORICAL_SESSION_WRITER_TEST else []
)

# ------------------------------------------------------------
# 3. Write each CME trading session to its own Parquet file.
# ------------------------------------------------------------

for session_date, session_frame in session_groups:
    session_frame = (
        session_frame
        .sort_values(["timestamp_second", "instrument_id"])
        .reset_index(drop=True)
    )

    output_path = (
        session_test_dir
        / f"MES_1s_{session_date}.parquet"
    )

    if output_path.exists():
        raise FileExistsError(f"Historical test target already exists: {output_path}")
    session_test_dir.mkdir(parents=True, exist_ok=True)
    session_frame.to_parquet(
        output_path,
        engine="pyarrow",
        compression="zstd",
        index=False,
    )

    written_files.append(
        (session_date, output_path, session_frame)
    )

    print(
        f"Wrote {session_date}: "
        f"{len(session_frame):,} rows -> "
        f"{output_path.name}"
    )

# ------------------------------------------------------------
# 4. Reload every saved session and verify that no values were
#    changed by the Parquet storage round trip.
# ------------------------------------------------------------

all_sessions_passed = True
reloaded_frames = []

print()
print("PARQUET RELOAD VALIDATION")
print("-------------------------")

for session_date, output_path, original_frame in written_files:
    reloaded_frame = pd.read_parquet(
        output_path,
        engine="pyarrow",
    )

    # Compare every stored value. Minor pandas dtype differences are
    # allowed because the underlying information is what matters here.
    pd.testing.assert_frame_equal(
        original_frame,
        reloaded_frame,
        check_dtype=False,
    )

    duplicate_rows = reloaded_frame.duplicated(
        subset=["timestamp_second", "instrument_id"]
    ).sum()

    session_passed = (
        len(original_frame) == len(reloaded_frame)
        and duplicate_rows == 0
        and reloaded_frame.columns.tolist()
        == original_frame.columns.tolist()
    )

    all_sessions_passed = (
        all_sessions_passed
        and session_passed
    )

    reloaded_frames.append(reloaded_frame)

    print(
        f"{session_date}: "
        f"{'PASS' if session_passed else 'FAIL'} "
        f"| rows={len(reloaded_frame):,} "
        f"| duplicates={duplicate_rows}"
    )

# ------------------------------------------------------------
# 5. Recombine the saved sessions and compare them with the
#    complete input dataset.
# ------------------------------------------------------------

if not RUN_HISTORICAL_SESSION_WRITER_TEST:
    print("Historical persistent session-writer test skipped.")
else:
    reloaded_all_sessions = (
        pd.concat(reloaded_frames, ignore_index=True)
        .sort_values(["timestamp_second", "instrument_id"])
        .reset_index(drop=True)
    )
    expected_all_sessions = (
        session_writer_data
        .sort_values(["timestamp_second", "instrument_id"])
        .reset_index(drop=True)
    )
    pd.testing.assert_frame_equal(
        expected_all_sessions, reloaded_all_sessions, check_dtype=False
    )
    print()
    print(f"Files written: {len(written_files)}")
    print(f"Reloaded rows: {len(reloaded_all_sessions):,}")
    print(f"Columns preserved: {len(reloaded_all_sessions.columns)}")
    print("All stored values match: True")
    print("Session writer test passed:", all_sessions_passed)

## Historical v1 production record — preserved, not executable

`full_history_v1` was built before the event-time audit established that the
production helper had used the Databento `ts_recv` index. It remains unchanged
for provenance and must not be used for feature engineering. The authoritative
event-time implementation and deliberately gated v2 command appear at the end
of this notebook.

In [63]:
# Historical v1 implementation note
#
# The former in-notebook production functions are intentionally superseded by
# src/trade_processing_v2.py. Keeping one authoritative implementation prevents
# prototype and production timestamp policies from drifting again.
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from trade_processing_v2 import (
    SCHEMA_VERSION, TIMESTAMP_POLICY, ONE_SECOND_COLUMNS, ONE_SECOND_ARROW_SCHEMA,
    add_trade_context, assert_event_time_order, apply_tick_rule_v1,
    split_complete_seconds, prepare_complete_trades, aggregate_to_one_second,
    add_contract_change_flag, enforce_one_second_schema, validate_one_second_session,
    process_full_history_event_time, audit_event_time_dataset,
)

print(f'Authoritative schema: {SCHEMA_VERSION}')
print(f'Timestamp policy: {TIMESTAMP_POLICY}')
print(f'Permanent columns: {len(ONE_SECOND_COLUMNS)}')

In [64]:
# full_history_v1 is deliberately not runnable from this notebook.
raise RuntimeError(
    'full_history_v1 is preserved receive-time provenance. Use the gated v2 '
    'production cell at the end of this notebook after code review.'
)

## Authoritative v2 event-time implementation

The source module below is intentionally small and mirrors the validated v1
architecture. It is the single implementation used by the tests, production
command, and post-run audit. Centralizing it prevents the earlier mistake where
the prototype used `ts_event` but the production helper used the `ts_recv`
index.

The chronological-order check is essential: holding a final second and
flushing a session are only valid when future records cannot belong to an
earlier event-time second or session. Equal event timestamps are permitted and
retain their DBN order for deterministic OHLC and tick-rule behavior.

In [1]:
# Import the authoritative v2 processor without starting a production run.
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from trade_processing_v2 import (
    SCHEMA_VERSION, TIMESTAMP_POLICY, ONE_SECOND_ARROW_SCHEMA,
    apply_tick_rule_v1, prepare_complete_trades, aggregate_to_one_second,
    add_contract_change_flag, enforce_one_second_schema,
    process_full_history_event_time, audit_event_time_dataset,
)

print(SCHEMA_VERSION)
print(TIMESTAMP_POLICY)
print(f'Fixed Arrow fields: {len(ONE_SECOND_ARROW_SCHEMA)}')

mes_1s_v2_event_time_33
Databento ts_event (exchange event time), stored in UTC
Fixed Arrow fields: 33


### V2 state and price-movement tests

These compact tests preserve the important behavior demonstrated earlier while
also checking the corrected no-prior-price rule. A first trade or a trade after
a contract roll has no comparable predecessor: it remains direction-unknown
for the tick rule and contributes to none of the three price-movement counts.
A same-price trade only means that a real, same-contract predecessor exists.

In [2]:
# Tiny deterministic test: tick state, contract reset, and no-prior-price semantics.
import pandas as pd

test = pd.DataFrame({
    'timestamp': pd.to_datetime([
        '2026-09-08 22:00:00+00:00', '2026-09-08 22:00:00.1+00:00',
        '2026-09-08 22:00:00.2+00:00', '2026-09-08 22:00:01+00:00',
    ], format='ISO8601'),
    'timestamp_second': pd.to_datetime([
        '2026-09-08 22:00:00+00:00', '2026-09-08 22:00:00+00:00',
        '2026-09-08 22:00:00+00:00', '2026-09-08 22:00:01+00:00',
    ]),
    'session_date': [pd.Timestamp('2026-09-09').date()] * 4,
    'instrument_id': [1, 1, 1, 2], 'contract': ['A', 'A', 'A', 'B'],
    'price': [100.0, 100.25, 100.25, 101.0], 'size': [1, 1, 1, 1],
    'side': ['A', 'B', 'B', 'B'],
})
prepared, *_ = prepare_complete_trades(test)
assert prepared['inferred_direction'].tolist() == [0, 1, 1, 0]
assert prepared[['uptick', 'downtick', 'same_price']].values.tolist() == [[0, 0, 0], [1, 0, 0], [0, 0, 1], [0, 0, 0]]

seconds = aggregate_to_one_second(prepared)
seconds, _ = add_contract_change_flag(seconds)
seconds = enforce_one_second_schema(seconds)
assert len(seconds.columns) == 33
assert seconds['uptick_count'].astype('int64').sum() == 1
assert seconds['same_price_count'].astype('int64').sum() == 1
print('V2 tick, reset, movement, and fixed-schema test: PASS')

V2 tick, reset, movement, and fixed-schema test: PASS


## Corrected full-history production command — intentionally gated

V2 production completed successfully with 99,319,450 raw trades, 298,546,252
raw volume, 13,165,975 one-second rows, and 242 session files. The saved
default remains `RUN_FULL_HISTORY_V2 = False`: this cell is retained only for
an intentional future rebuild to a new empty output directory. It refuses a
non-empty target and cannot overwrite `full_history_v1`.

The completed output includes `run_manifest.json` and `session_manifest.json`,
which record timestamp policy, schema version, source coverage, row/trade/volume
totals, per-session contracts and rolls, nominal-window coverage, and degraded
status.

In [4]:
# Keep the expensive full-history production build disabled by default.
RUN_FULL_HISTORY_V2 = False

raw_history_path = project_root / 'data' / 'mes_trades_2025-10-07_to_2026-09-11.dbn'
v2_output_dir = project_root / 'data' / 'processed' / '1s' / 'full_history_v2_event_time'

if RUN_FULL_HISTORY_V2:
    v2_result = process_full_history_event_time(
        raw_path=raw_history_path,
        output_dir=v2_output_dir,
        chunk_size=500_000,
    )
    print(v2_result)
else:
    print('Not run. Set RUN_FULL_HISTORY_V2 = True only for the intentional v2 build.')

{'raw_trades': 99319450, 'raw_volume': 298546252, 'second_rows': 13165975, 'sessions_written': 242, 'output_dir': '/Users/marcusromeo/Desktop/mes-trading-project/data/processed/1s/full_history_v2_event_time'}


## Post-run event-time audit — intentionally gated

The independent post-run audit passed for the completed V2 dataset. It rescans the raw DBN in
bounded chunks and compares raw event-time trade count, total volume, and active
`(timestamp_second, instrument_id)` rows with the stored v2 files. It also
checks every session’s fixed schema, ordering, CME date, unique keys, contract
mapping/rolls, full session-manifest facts, source-boundary partial sessions, and
known degraded sessions. It is designed for an 8 GB Mac: only one raw chunk or one session
Parquet file is held at a time.

The saved default remains `RUN_POST_RUN_AUDIT = False`; rerun it only after an intentional future V2 build or a manifest/audit-code change.

In [5]:
# Keep the expensive post-run audit disabled by default.
RUN_POST_RUN_AUDIT = False

if RUN_POST_RUN_AUDIT:
    v2_audit = audit_event_time_dataset(
        output_dir=v2_output_dir,
        raw_path=raw_history_path,
        chunk_size=500_000,
    )
    print(v2_audit)
else:
    print('Not run. Set RUN_POST_RUN_AUDIT = True only after a completed v2 build.')

{'files': 242, 'sessions': 242, 'rows': 13165975, 'trade_count': 99319450, 'volume': 298546252, 'event_time_second_rows': 13165975, 'contract_changes': [('2025-12-17 00:00:00+00:00', 42003800), ('2026-03-18 00:00:00+00:00', 42005163), ('2026-06-17 00:00:00+00:00', 42003239)], 'partial_sessions': ['2025-10-07', '2026-09-11'], 'degraded_sessions': ['2025-11-28', '2026-03-16', '2026-03-17', '2026-04-10', '2026-05-25', '2026-07-30', '2026-07-31'], 'schema_version': 'mes_1s_v2_event_time_33', 'passed': True}


## Decision and next step

`full_history_v1` remains a receive-time provenance artifact and is not approved
for modeling. `full_history_v2_event_time` is the approved historical research
foundation: its production run and independent audit both passed. The next stage
is feature engineering with explicit decision-time rules and deliberate handling
of partial, degraded, holiday, and contract-roll sessions. Native aggressor side
remains a historical benchmark, not a production feature.